# Ground-truth Generators

- $v_1 = \partial_t$
- $v_2 = u \partial_u$
- $v_3 = e^t \partial_x$
- $v_4 = e^{-t}[\partial_x+2xu\partial_u]$
- $v_5 = e^{2t}[\partial_t + x\partial_x - u\partial_u]$
- $v_6 = e^{-2t}[-\partial_t + x\partial_x + 2x^2u\partial_u]$

$v_1, v_3, v_5$ are SDE sym generators

# SDE Symmetry

##Imports & Downloads

In [ ]:
!pip install dm-haiku

In [ ]:
# Install (if needed)
# pip install --quiet jax jaxlib optax dm-haiku

import math
import functools
import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import optax

jax.config.update("jax_enable_x64", True)

import itertools
from functools import partial

import matplotlib.pyplot as plt

Array = jnp.ndarray

In [ ]:
print(jnp.array([0.]).dtype)   # should print float64


## Neural SDE

In [ ]:
# @title Neural SDE surrogate for dx_t = x_t dt + σ₀ dW_t (1D linear SDE example) + plots  (Option A: IID increments on bounded x-domain)
# Runs as-is in Google Colab (JAX + optax)
#
# MINIMUM downstream-impact changes:
#   - keep exports: cfg, params_sde, in_norm_sde, loss_history_sde (UNCHANGED)
#   - keep helper APIs: init_mlp_params, mlp_forward, f_sigma_hat (UNCHANGED)
#   - keep variable names used later: t, x, X_raw, dX, in_norm (UNCHANGED)
#   - only replace trajectory simulation with IID increment sampling on a bounded x-range

import math
from dataclasses import dataclass

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

try:
    import optax
    OPTAX_AVAILABLE = True
except Exception:
    OPTAX_AVAILABLE = False
    raise RuntimeError("This cell expects optax to be available in Colab.")

# Use float64
jax.config.update("jax_enable_x64", True)

# -------------------------------------------------------------------
# 0. Config and helpers
# -------------------------------------------------------------------

@dataclass
class CFG:
    # Ground-truth SDE: dx_t = x_t dt + σ0 dW_t
    sigma0: float = 1.0

    # Time grid
    T: float = 5.0          # final time
    dt: float = 0.01        # time step
    n_traj: int = 512       # used ONLY to match the old sample count: B = n_traj * (T/dt)

    # NN + training
    hidden: int = 64
    steps: int = 2000
    batch_size: int = 4096
    lr: float = 3e-3
    weight_decay: float = 1e-6
    sigma_min: float = 1e-3  # lower bound on σ̂ for stability

    # ---- Downstream-compat extras (defaults; do NOT change behavior) ----
    x_max: float = 2.0               # used by some plotting/animation helpers (getattr-safe)
    x0_mode: str = "point"           # used by analytic-FP compare cell
    x0_value: float = 0.0            # matches x(0)=0 below
    x0_mean: float = 0.0             # unused unless x0_mode="normal"
    x0_std: float = 1.0              # unused unless x0_mode="normal"
    x0_low: float = -1.0             # unused unless x0_mode="uniform"
    x0_high: float = 1.0             # unused unless x0_mode="uniform"
    x0_points: tuple = (0.0,)        # unused unless x0_mode in ("mixture_points","grid_points")

cfg = CFG()

key_main = jax.random.PRNGKey(0)

# -------------------------------------------------------------------
# 1. Data generation (Option A): IID increments on a bounded x-domain
# -------------------------------------------------------------------
# We keep `t` and `x` variables for downstream compatibility/prints,
# but they are now "grid + sampled x cloud" rather than simulated trajectories.

def make_iid_increment_dataset(key, cfg: CFG, x_train_max: float):
    """
    Option A dataset:
      Sample (t, x) from a chosen domain and generate Δx from the exact increment law:
        Δx | (t,x) ~ Normal( x * dt , (sigma0^2) * dt )

    Returns (and keeps old names):
      t : (N+1,) time grid (for compatibility)
      x : (n_traj, N+1) a reshaped view of sampled x's (for compatibility / printing)
      X_raw : (B,2) inputs [t_n, x_n]
      dX : (B,1)  targets Δx_n
    """
    dt = float(cfg.dt)
    N = int(cfg.T / dt)                 # number of increments
    B = int(cfg.n_traj) * N             # match old dataset size exactly

    # deterministic time grid kept for compatibility (used by later plotting code sometimes)
    t_grid = jnp.linspace(0.0, cfg.T, N + 1, dtype=jnp.float64)  # (N+1,)

    key_t, key_x, key_eps = jax.random.split(key, 3)

    # sample t from the *left-endpoint* grid {0, dt, ..., T-dt} (like trajectory increments)
    t_left_grid = jnp.linspace(0.0, cfg.T - cfg.dt, N, dtype=jnp.float64)  # (N,)
    idx = jax.random.randint(key_t, (B,), 0, N)
    t_samp = t_left_grid[idx].reshape(B, 1)  # (B,1)

    # bounded x domain
    x_samp = jax.random.uniform(
        key_x, (B, 1),
        minval=-x_train_max,
        maxval= x_train_max,
        dtype=jnp.float64
    )  # (B,1)

    # exact increment law for dx = x dt + sigma0 dW
    eps = jax.random.normal(key_eps, (B, 1), dtype=jnp.float64)
    dx = x_samp * cfg.dt + cfg.sigma0 * jnp.sqrt(cfg.dt) * eps  # (B,1)

    X_raw = jnp.concatenate([t_samp, x_samp], axis=1)  # (B,2)
    dX = dx                                           # (B,1)

    # For downstream compatibility, also provide an "x array" of shape (n_traj, N+1).
    # This is NOT a true trajectory; it's just a reshaping of sampled x values.
    # We'll fill the last column with zeros to keep (N+1) columns.
    x_mat = x_samp.reshape(cfg.n_traj, N)  # (n_traj, N)
    x_full = jnp.concatenate(
        [x_mat, jnp.zeros((cfg.n_traj, 1), dtype=jnp.float64)],
        axis=1
    )  # (n_traj, N+1)

    return t_grid, x_full, X_raw, dX

# --------- USER CONTROL: training x-domain size (Option A) -------------------
# Keep it modest (e.g. 3, 5, 10) to avoid huge |x| during surrogate training.
x_train_max = float(getattr(cfg, "x_max", 2.0))  # default uses cfg.x_max (keeps API stable)

t, x, X_raw, dX = make_iid_increment_dataset(key_main, cfg, x_train_max=x_train_max)

print("Data shapes: t =", t.shape, ", x =", x.shape)
print("Surrogate-training x range:", float(x[:, :-1].min()), float(x[:, :-1].max()))
print("Surrogate-training x mean/std:", float(x[:, :-1].mean()), float(x[:, :-1].std()))

print("Increment dataset shapes: X_raw =", X_raw.shape, ", dX =", dX.shape)
print("Surrogate-training x range (from X_raw):",
      float(X_raw[:, 1].min()), float(X_raw[:, 1].max()))

# -------------------------------------------------------------------
# 3. Normalization (z-score) for inputs (t,x)
# -------------------------------------------------------------------

class Normalizer:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

def fit_normalizer(X):
    return Normalizer(jnp.mean(X, axis=0), jnp.std(X, axis=0))

in_norm = fit_normalizer(X_raw)
X = in_norm(X_raw)  # normalized inputs
print("Input mean ~", X.mean(0), "std ~", X.std(0))

# -------------------------------------------------------------------
# 4. MLP model: (t,x) -> (f̂, g) with σ̂ = softplus(g) + σ_min
# -------------------------------------------------------------------

def glorot(key, fan_in, fan_out):
    lim = math.sqrt(6.0 / (fan_in + fan_out))
    return jax.random.uniform(key, (fan_in, fan_out), minval=-lim, maxval=lim)

def init_mlp_params(key, sizes):
    """
    sizes: list of layer widths [in_dim, h1, h2, ..., out_dim]
    returns: list of {'W', 'b'} dicts
    """
    keys = jax.random.split(key, len(sizes) - 1)
    params = []
    for k, (m, n) in zip(keys, zip(sizes[:-1], sizes[1:])):
        params.append({
            "W": glorot(k, m, n),
            "b": jnp.zeros((n,), dtype=jnp.float64),
        })
    return params

def mlp_forward(params, x, activation="tanh"):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            if activation == "tanh":
                h = jnp.tanh(h)
            elif activation == "relu":
                h = jax.nn.relu(h)
            elif activation == "swish":
                h = jax.nn.silu(h)   # <-- NEW
            elif activation == "gelu":
                h = jax.nn.gelu(h)   # <-- NEW
            elif activation == "sin":
                h = jnp.sin(h)       # optional, useful later
            else:
                raise ValueError(f"Unknown activation {activation}")
    return h

def f_sigma_hat(params, tx_norm, activation="tanh", sigma_min=1e-3):
    """
    tx_norm: (..., 2) normalized [t,x].
    Returns:
      f_hat: drift estimate
      sigma_hat: diffusion estimate (positive)
    """
    out = mlp_forward(params, tx_norm, activation=activation)  # (..., 2)
    f_hat = out[..., 0:1]           # (...,1)
    g = out[..., 1:2]               # (...,1)
    sigma_hat = jax.nn.softplus(g) + sigma_min
    return f_hat, sigma_hat

# Init model params
key_main, k_model = jax.random.split(key_main, 2)
params_sde = init_mlp_params(k_model, sizes=[2, cfg.hidden, cfg.hidden, 2])

# -------------------------------------------------------------------
# 5. Increment-based NLL loss
# -------------------------------------------------------------------

N = X.shape[0]
print("Total samples:", N)

def make_increment_loss(dt, sigma_min, weight_decay):

    def loss_fn(params, xb, dxb):
        """
        xb: (B,2) normalized inputs [t,x]
        dxb: (B,1) increments Δx
        """
        f_hat, sigma_hat = f_sigma_hat(params, xb, activation="tanh", sigma_min=sigma_min)
        # mean and variance for increments
        mu = f_hat * dt                      # (B,1)
        var = (sigma_hat ** 2) * dt          # (B,1)

        # Negative log-likelihood (up to constant 0.5 log(2π))
        residual = dxb - mu
        ell = (residual ** 2) / (2.0 * var) + 0.5 * jnp.log(var + 1e-12)

        # Average over minibatch
        nll = jnp.mean(ell)

        # Weight decay
        def l2_tree(p):
            return sum([jnp.sum(v**2) for v in jax.tree_util.tree_leaves(p)])

        reg = weight_decay * l2_tree(params)
        return nll + reg

    return loss_fn

loss_fn = make_increment_loss(cfg.dt, cfg.sigma_min, cfg.weight_decay)
loss_fn_jit = jax.jit(loss_fn)

# -------------------------------------------------------------------
# 6. Training loop (Adam + minibatches) + record loss for plotting
# -------------------------------------------------------------------

if not OPTAX_AVAILABLE:
    raise RuntimeError("optax is required for this training loop.")

optimizer = optax.adam(cfg.lr)
opt_state = optimizer.init(params_sde)

rng_np = np.random.default_rng(0)

@jax.jit
def train_step(params, opt_state, xb, dxb):
    def _loss(p):
        return loss_fn(p, xb, dxb)
    val, grads = jax.value_and_grad(_loss)(params)
    updates, opt_state_new = optimizer.update(grads, opt_state, params)
    params_new = optax.apply_updates(params, updates)
    return params_new, opt_state_new, val

def sample_minibatch(X, dX, batch_size):
    N = X.shape[0]
    if batch_size >= N:
        idx = np.arange(N)
    else:
        idx = rng_np.choice(N, size=batch_size, replace=False)
    xb = X[idx]
    dxb = dX[idx]
    return xb, dxb

print_every = 100
loss_history = []

for step in range(1, cfg.steps + 1):
    xb, dxb = sample_minibatch(X, dX, cfg.batch_size)
    xb = jnp.asarray(xb)
    dxb = jnp.asarray(dxb)

    params_sde, opt_state, loss_val = train_step(params_sde, opt_state, xb, dxb)
    loss_float = float(loss_val)
    loss_history.append(loss_float)

    if step % print_every == 0 or step == 1 or step == cfg.steps:
        print(f"step {step:5d}/{cfg.steps} | NLL+reg = {loss_float:.6e}")

print("\nTraining finished.")

# -------------------------------------------------------------------
# 7. Plot training loss curve
# -------------------------------------------------------------------

steps_arr = np.arange(1, cfg.steps + 1)

plt.figure(figsize=(6, 4))
plt.plot(steps_arr, loss_history, lw=2)
plt.xlabel("Training step")
plt.ylabel("NLL + reg")
plt.title("Training loss for neural SDE surrogate")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 8. Quick diagnostics: learned f̂ and σ̂ on a small grid
# -------------------------------------------------------------------

def eval_model(params, t_vals, x_vals):
    TT, XX = jnp.meshgrid(t_vals, x_vals, indexing="ij")
    TX = jnp.stack([TT.ravel(), XX.ravel()], axis=-1)   # (B,2)
    TX_norm = in_norm(TX)
    f_hat, sigma_hat = f_sigma_hat(params, TX_norm, activation="tanh", sigma_min=cfg.sigma_min)
    return TT, XX, f_hat.reshape(TT.shape), sigma_hat.reshape(TT.shape)

t_eval = jnp.linspace(0.0, cfg.T, 21)
x_eval = jnp.linspace(-2.0, 2.0, 21)

TT, XX, F_hat_grid, Sigma_hat_grid = eval_model(params_sde, t_eval, x_eval)

print("\nGround-truth σ0 =", cfg.sigma0)
print("Mean σ̂ over eval grid:", float(Sigma_hat_grid.mean()))
print("Std  σ̂ over eval grid:", float(Sigma_hat_grid.std()))
print("Mean |f̂| over eval grid:", float(jnp.abs(F_hat_grid).mean()))

# Heatmaps of f̂ and σ̂
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

# σ̂(t,x)
im0 = axes[0].imshow(
    np.asarray(Sigma_hat_grid),
    origin="lower",
    extent=(float(x_eval.min()), float(x_eval.max()),
            float(t_eval.min()), float(t_eval.max())),
    aspect="auto"
)
axes[0].set_title(r"$\hat{\sigma}(t,x)$")
axes[0].set_xlabel("x")
axes[0].set_ylabel("t")
fig.colorbar(im0, ax=axes[0], shrink=0.9)

# f̂(t,x)
im1 = axes[1].imshow(
    np.asarray(F_hat_grid),
    origin="lower",
    extent=(float(x_eval.min()), float(x_eval.max()),
            float(t_eval.min()), float(t_eval.max())),
    aspect="auto"
)
axes[1].set_title(r"$\hat{f}(t,x)$")
axes[1].set_xlabel("x")
axes[1].set_ylabel("t")
fig.colorbar(im1, ax=axes[1], shrink=0.9)

plt.show()

# -------------------------------------------------------------------
# 9. Export trained objects to globals for later stages (as expected downstream)
# -------------------------------------------------------------------

globals().update({
    "cfg": cfg,
    "params_sde": params_sde,
    "in_norm_sde": in_norm,
    "loss_history_sde": list(zip(steps_arr, loss_history)),
})

print("\nExported: params_sde, in_norm_sde, cfg, loss_history_sde.")


### Alternative Surrogate

In [ ]:
# @title Neural SDE surrogate (DATA-ONLY training from trajectory increments) + plots
# Runs as-is in Google Colab (JAX + optax)
#
# MINIMUM downstream-impact changes:
#   - keep exports: cfg, params_sde, in_norm_sde, loss_history_sde (UNCHANGED)
#   - keep helper APIs: init_mlp_params, mlp_forward, f_sigma_hat (UNCHANGED)
#   - keep variable names used later: t, x, X_raw, dX, in_norm (UNCHANGED)
#   t_data: (N+1,)
#   x_data: (n_traj, N+1)
# before running this cell. Otherwise the cell generates a demo dataset.
#
# NOTE: cfg.sigma0 is kept ONLY for downstream plotting compatibility
#       and is set from data via a crude empirical estimate.

import math
import importlib
import subprocess
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

# --------- Ensure deps in Colab ----------
def _ensure(pkg):
    try:
        importlib.import_module(pkg)
        return True
    except Exception:
        return False

if not _ensure("optax"):
    subprocess.check_call(["bash", "-lc", "pip -q install optax"])
if not _ensure("jax"):
    subprocess.check_call(["bash", "-lc", "pip -q install 'jax[cpu]'"])

import jax
import jax.numpy as jnp
import optax

jax.config.update("jax_enable_x64", True)

# -------------------------------------------------------------------
# 0. Config and helpers
# -------------------------------------------------------------------

@dataclass
class CFG:
    # Time grid
    T: float = 5.0
    dt: float = 0.01
    n_traj: int = 512

    # NN + training
    hidden: int = 64
    steps: int = 2500
    batch_size: int = 4096
    lr: float = 3e-3
    weight_decay: float = 1e-6
    sigma_min: float = 1e-3

    # Training stability
    grad_clip: float = 1.0
    val_frac: float = 0.10
    patience: int = 200  # early stopping patience (in steps)

    # ---- Downstream-compat extras ----
    x_max: float = 2.0
    x0_mode: str = "point"
    x0_value: float = 0.0
    x0_mean: float = 0.0
    x0_std: float = 1.0
    x0_low: float = -1.0
    x0_high: float = 1.0
    x0_points: tuple = (0.0,)


    # NOT used for training; set after dataset build from data.
    sigma0: float = 1.0  # will be overwritten by empirical estimate

    # ---- Demo-data only ----
    demo_sigma0: float = 1.0
    demo_x0: float = 0.0

cfg = CFG()
key_main = jax.random.PRNGKey(0)

# -------------------------------------------------------------------
# 1. DATASET: build (t_n, x_n) -> Δx_n purely from trajectory data
# -------------------------------------------------------------------

def _build_increment_dataset_from_trajectories(t_path, x_paths):
    """
    t_path: (N+1,)
    x_paths: (n_traj, N+1)
    Returns:
      t : (N+1,)
      x : (n_traj, N+1)
      X_raw : (B,2) inputs [t_n, x_n]
      dX : (B,1) targets Δx_n = x_{n+1} - x_n
    """
    t_path = jnp.asarray(t_path, dtype=jnp.float64)
    x_paths = jnp.asarray(x_paths, dtype=jnp.float64)

    Np1 = t_path.shape[0]
    N = Np1 - 1
    n_traj = x_paths.shape[0]

    t_left = t_path[:-1]                             # (N,)
    x_left = x_paths[:, :-1]                         # (n_traj, N)
    dx = x_paths[:, 1:] - x_paths[:, :-1]            # (n_traj, N)

    TT = jnp.broadcast_to(t_left[None, :], (n_traj, N)).reshape(-1, 1)  # (B,1)
    XX = x_left.reshape(-1, 1)                                         # (B,1)
    dX = dx.reshape(-1, 1)                                             # (B,1)

    X_raw = jnp.concatenate([TT, XX], axis=1)                          # (B,2)
    return t_path, x_paths, X_raw, dX

def _simulate_demo_trajectories(key, cfg: CFG):
    dt = float(cfg.dt)
    N = int(cfg.T / dt)
    t_path = jnp.linspace(0.0, cfg.T, N + 1, dtype=jnp.float64)

    eps = jax.random.normal(key, (cfg.n_traj, N), dtype=jnp.float64)

    x0 = jnp.full((cfg.n_traj,), float(cfg.demo_x0), dtype=jnp.float64)
    x_list = [x0]
    x_curr = x0
    for n in range(N):
        drift = x_curr
        x_next = x_curr + drift * dt + float(cfg.demo_sigma0) * jnp.sqrt(dt) * eps[:, n]
        x_list.append(x_next)
        x_curr = x_next

    x_paths = jnp.stack(x_list, axis=1)
    return t_path, x_paths

# Use user-provided data if present; else demo-generate
if ("t_data" in globals()) and ("x_data" in globals()):
    t, x, X_raw, dX = _build_increment_dataset_from_trajectories(globals()["t_data"], globals()["x_data"])
    print("Using provided data: t_data/x_data")
else:
    t_demo, x_demo = _simulate_demo_trajectories(key_main, cfg)
    t, x, X_raw, dX = _build_increment_dataset_from_trajectories(t_demo, x_demo)
    print("No t_data/x_data found: using DEMO simulated trajectories (training is still data-only).")

# ---- Data-only empirical sigma reference for plotting compatibility ----
# crude: sigma ≈ sqrt(Var(Δx)/dt)
dx_var = float(jnp.var(dX))
cfg.sigma0 = cfg.demo_sigma0

print("Data shapes: t =", t.shape, ", x =", x.shape)
print("Increment dataset shapes: X_raw =", X_raw.shape, ", dX =", dX.shape)
print("x range:", float(x.min()), float(x.max()), "| dx std:", float(dX.std()))
print("cfg.sigma0 (empirical, for plotting only) =", cfg.sigma0)

# -------------------------------------------------------------------
# 3. Normalization (z-score) for inputs (t,x)
# -------------------------------------------------------------------

class Normalizer:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

def fit_normalizer(X):
    return Normalizer(jnp.mean(X, axis=0), jnp.std(X, axis=0))

in_norm = fit_normalizer(X_raw)
X = in_norm(X_raw)
print("Input mean ~", X.mean(0), "std ~", X.std(0))

# -------------------------------------------------------------------
# 4. MLP model: (t,x) -> (f̂, g) with σ̂ = softplus(g) + σ_min
# -------------------------------------------------------------------

def glorot(key, fan_in, fan_out):
    lim = math.sqrt(6.0 / (fan_in + fan_out))
    return jax.random.uniform(key, (fan_in, fan_out), minval=-lim, maxval=lim, dtype=jnp.float64)

def init_mlp_params(key, sizes):
    keys = jax.random.split(key, len(sizes) - 1)
    params = []
    for k, (m, n) in zip(keys, zip(sizes[:-1], sizes[1:])):
        params.append({
            "W": glorot(k, m, n),
            "b": jnp.zeros((n,), dtype=jnp.float64),
        })
    return params

def mlp_forward(params, x, activation="tanh"):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            if activation == "tanh":
                h = jnp.tanh(h)
            elif activation == "relu":
                h = jax.nn.relu(h)
            else:
                raise ValueError(f"Unknown activation {activation}")
    return h

def f_sigma_hat(params, tx_norm, activation="tanh", sigma_min=1e-3):
    out = mlp_forward(params, tx_norm, activation=activation)  # (..., 2)
    f_hat = out[..., 0:1]
    g = out[..., 1:2]
    sigma_hat = jax.nn.softplus(g) + sigma_min
    return f_hat, sigma_hat

key_main, k_model = jax.random.split(key_main, 2)
params_sde = init_mlp_params(k_model, sizes=[2, cfg.hidden, cfg.hidden, 2])

# -------------------------------------------------------------------
# 5. Increment-based Gaussian NLL loss (data-only)
# -------------------------------------------------------------------

def make_increment_loss(dt, sigma_min, weight_decay):
    def loss_fn(params, xb, dxb):
        f_hat, sigma_hat = f_sigma_hat(params, xb, activation="tanh", sigma_min=sigma_min)

        mu = f_hat * dt
        var = (sigma_hat ** 2) * dt
        var = jnp.maximum(var, 1e-12)

        residual = dxb - mu
        ell = (residual ** 2) / (2.0 * var) + 0.5 * jnp.log(var)

        nll = jnp.mean(ell)

        def l2_tree(p):
            return sum([jnp.sum(v**2) for v in jax.tree_util.tree_leaves(p)])
        reg = weight_decay * l2_tree(params)
        return nll + reg
    return loss_fn

loss_fn = make_increment_loss(cfg.dt, cfg.sigma_min, cfg.weight_decay)

# -------------------------------------------------------------------
# 6. Training loop (AdamW + grad clip + val split + early stopping)
# -------------------------------------------------------------------

N_total = X.shape[0]
rng_np = np.random.default_rng(0)

perm = rng_np.permutation(N_total)
n_val = int(cfg.val_frac * N_total)
val_idx = perm[:n_val]
train_idx = perm[n_val:]

X_train = X[train_idx]
dX_train = dX[train_idx]
X_val = X[val_idx]
dX_val = dX[val_idx]

print("Total samples:", N_total, "| train:", X_train.shape[0], "| val:", X_val.shape[0])

optimizer = optax.chain(
    optax.clip_by_global_norm(cfg.grad_clip),
    optax.adamw(learning_rate=cfg.lr, weight_decay=0.0)
)
opt_state = optimizer.init(params_sde)

@jax.jit
def train_step(params, opt_state, xb, dxb):
    def _loss(p):
        return loss_fn(p, xb, dxb)
    val, grads = jax.value_and_grad(_loss)(params)
    updates, opt_state_new = optimizer.update(grads, opt_state, params)
    params_new = optax.apply_updates(params, updates)
    return params_new, opt_state_new, val

@jax.jit
def eval_loss(params, xb, dxb):
    return loss_fn(params, xb, dxb)

def sample_minibatch(Xarr, dXarr, batch_size):
    N = Xarr.shape[0]
    if batch_size >= N:
        idx = np.arange(N)
    else:
        idx = rng_np.choice(N, size=batch_size, replace=False)
    return jnp.asarray(np.asarray(Xarr)[idx]), jnp.asarray(np.asarray(dXarr)[idx])

print_every = 100
loss_history = []
val_history = []

best_params = params_sde
best_val = float("inf")
bad_steps = 0

for step in range(1, cfg.steps + 1):
    xb, dxb = sample_minibatch(X_train, dX_train, cfg.batch_size)
    params_sde, opt_state, loss_val = train_step(params_sde, opt_state, xb, dxb)

    if step % 10 == 0 or step == 1:
        v = float(eval_loss(params_sde, X_val, dX_val))
        val_history.append(v)
        if v < best_val - 1e-6:
            best_val = v
            best_params = params_sde
            bad_steps = 0
        else:
            bad_steps += 10
    else:
        v = None

    loss_float = float(loss_val)
    loss_history.append(loss_float)

    if step % print_every == 0 or step == 1 or step == cfg.steps:
        msg = f"step {step:5d}/{cfg.steps} | train NLL+reg = {loss_float:.6e}"
        if v is not None:
            msg += f" | val = {v:.6e} | best_val = {best_val:.6e}"
        print(msg)

    if bad_steps >= cfg.patience:
        print(f"Early stopping at step {step} (no val improvement for ~{cfg.patience} steps).")
        break

params_sde = best_params
print("\nTraining finished. Using best validation params.")

# -------------------------------------------------------------------
# 7. Plot training curves
# -------------------------------------------------------------------

steps_arr = np.arange(1, len(loss_history) + 1)

plt.figure(figsize=(6, 4))
plt.plot(steps_arr, loss_history, lw=2, label="train")
if len(val_history) > 0:
    val_steps = np.array([s for s in range(1, len(loss_history) + 1) if (s % 10 == 0 or s == 1)])
    plt.plot(val_steps[:len(val_history)], val_history, lw=2, label="val")
plt.xlabel("Training step")
plt.ylabel("NLL + reg")
plt.title("Training loss for neural SDE surrogate (data-only)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 8. Quick diagnostics: learned f̂ and σ̂ on a small grid
# -------------------------------------------------------------------

def eval_model(params, t_vals, x_vals):
    TT, XX = jnp.meshgrid(t_vals, x_vals, indexing="ij")
    TX = jnp.stack([TT.ravel(), XX.ravel()], axis=-1)
    TX_norm = in_norm(TX)
    f_hat, sigma_hat = f_sigma_hat(params, TX_norm, activation="tanh", sigma_min=cfg.sigma_min)
    return TT, XX, f_hat.reshape(TT.shape), sigma_hat.reshape(TT.shape)

t_eval = jnp.linspace(0.0, cfg.T, 21)
x_eval = jnp.linspace(float(x.min()), float(x.max()), 21)

TT, XX, F_hat_grid, Sigma_hat_grid = eval_model(params_sde, t_eval, x_eval)

print("cfg.sigma0 (plot reference) =", cfg.sigma0)
print("Mean σ̂ over eval grid:", float(Sigma_hat_grid.mean()))
print("Std  σ̂ over eval grid:", float(Sigma_hat_grid.std()))
print("Mean |f̂| over eval grid:", float(jnp.abs(F_hat_grid).mean()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

im0 = axes[0].imshow(
    np.asarray(Sigma_hat_grid),
    origin="lower",
    extent=(float(x_eval.min()), float(x_eval.max()),
            float(t_eval.min()), float(t_eval.max())),
    aspect="auto"
)
axes[0].set_title(r"$\hat{\sigma}(t,x)$")
axes[0].set_xlabel("x")
axes[0].set_ylabel("t")
fig.colorbar(im0, ax=axes[0], shrink=0.9)

im1 = axes[1].imshow(
    np.asarray(F_hat_grid),
    origin="lower",
    extent=(float(x_eval.min()), float(x_eval.max()),
            float(t_eval.min()), float(t_eval.max())),
    aspect="auto"
)
axes[1].set_title(r"$\hat{f}(t,x)$")
axes[1].set_xlabel("x")
axes[1].set_ylabel("t")
fig.colorbar(im1, ax=axes[1], shrink=0.9)

plt.show()

# -------------------------------------------------------------------
# 9. Export trained objects to globals for later stages (as expected downstream)
# -------------------------------------------------------------------

globals().update({
    "cfg": cfg,
    "params_sde": params_sde,
    "in_norm_sde": in_norm,
    "loss_history_sde": list(zip(np.arange(1, len(loss_history) + 1), loss_history)),
})

print("\nExported: params_sde, in_norm_sde, cfg, loss_history_sde.")


In [ ]:
# Animations: drift and diffusion vs x as time evolves (diffusion frame auto-fits all frames)
# Requires in scope:
#   - params_sde, in_norm_sde, f_sigma_hat, cfg
#   - t (optional; if missing, uses [0,cfg.T])

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

sigma_min = float(getattr(cfg, "sigma_min", 1e-3))

def eval_fx_sigma(params, in_norm, t0, x_min=-2.0, x_max=2.0, n_points=200):
    x_vals = jnp.linspace(x_min, x_max, n_points, dtype=jnp.float64)
    t_vals = jnp.full_like(x_vals, t0)
    TX = jnp.stack([t_vals, x_vals], axis=-1)
    TX_norm = in_norm(TX)
    f_hat, sigma_hat = f_sigma_hat(params, TX_norm, activation="tanh", sigma_min=sigma_min)
    return (np.asarray(x_vals),
            np.asarray(f_hat.squeeze(-1)),
            np.asarray(sigma_hat.squeeze(-1)))

# Time grid for frames
if "t" in globals():
    t_array = np.asarray(t)
    t_start, t_end = float(t_array[0]), float(t_array[-1])
else:
    t_start, t_end = 0.0, float(getattr(cfg, "T", 5.0))

t_frames = np.linspace(t_start, t_end, 100)

# x-range for plots
x_max_cfg = float(getattr(cfg, "x_max", 2.0))
x_min, x_max = -x_max_cfg, x_max_cfg

# ================== 1) Drift animation =======================================
fig_drift, ax_drift = plt.subplots(figsize=(6, 4))

line_drift, = ax_drift.plot([], [], lw=2, label=r"learned $\hat f_\theta(t,x)$")
line_drift_true, = ax_drift.plot([], [], lw=2, linestyle="--", color="k",
                                 label=r"ground truth $f(t,x)=x$")

ax_drift.set_xlim(x_min, x_max)
ax_drift.set_ylim(x_min, x_max)
ax_drift.set_xlabel("x")
ax_drift.set_ylabel("drift")
title_drift = ax_drift.set_title("")
ax_drift.grid(True, alpha=0.3)
ax_drift.legend(loc="upper left")

def init_drift():
    line_drift.set_data([], [])
    line_drift_true.set_data([], [])
    title_drift.set_text("")
    return line_drift, line_drift_true, title_drift

def update_drift(frame_idx):
    t0 = float(t_frames[frame_idx])
    x_vals, f_vals, _ = eval_fx_sigma(params_sde, in_norm_sde, t0, x_min=x_min, x_max=x_max)
    line_drift.set_data(x_vals, f_vals)
    line_drift_true.set_data(x_vals, x_vals)
    title_drift.set_text(rf"Drift $\hat f_\theta(t,x)$ at $t={t0:.2f}$")
    return line_drift, line_drift_true, title_drift

anim_drift = FuncAnimation(
    fig_drift,
    update_drift,
    init_func=init_drift,
    frames=len(t_frames),
    interval=80,
    blit=True,
)

plt.close(fig_drift)
display(HTML(anim_drift.to_jshtml()))

# ================== 2) Diffusion animation ===================================
fig_diff, ax_diff = plt.subplots(figsize=(6, 4))

line_diff, = ax_diff.plot([], [], lw=2, label=r"learned $\hat\sigma_\theta(t,x)$")
ax_diff.axhline(cfg.sigma0, color="k", linestyle="--",
                label=rf"ground truth $\sigma(t,x)=\sigma_0={cfg.sigma0}$")

ax_diff.set_xlim(x_min, x_max)

# ---- FIX: set y-limits to cover ALL frames, not just the first probe ----
# Sample a subset of frame times to estimate global min/max robustly.
# (100 frames * 200 points is fine too; this is just a bit lighter.)
probe_idx = np.linspace(0, len(t_frames) - 1, 25, dtype=int)
smins, smaxs = [], []
for i in probe_idx:
    _, _, s_vals = eval_fx_sigma(params_sde, in_norm_sde, float(t_frames[i]),
                                 x_min=x_min, x_max=x_max, n_points=200)
    smins.append(float(np.min(s_vals)))
    smaxs.append(float(np.max(s_vals)))

s_lo = min(min(smins), float(cfg.sigma0))
s_hi = max(max(smaxs), float(cfg.sigma0))

# add generous padding so curves never clip
pad = 0.15 * (s_hi - s_lo + 1e-6)
ax_diff.set_ylim(s_lo - pad, s_hi + pad)
# ------------------------------------------------------------------------

ax_diff.set_xlabel("x")
ax_diff.set_ylabel("diffusion")
title_diff = ax_diff.set_title("")
ax_diff.grid(True, alpha=0.3)
ax_diff.legend(loc="upper left")

def init_diff():
    line_diff.set_data([], [])
    title_diff.set_text("")
    return line_diff, title_diff

def update_diff(frame_idx):
    t0 = float(t_frames[frame_idx])
    x_vals, _, sigma_vals = eval_fx_sigma(params_sde, in_norm_sde, t0, x_min=x_min, x_max=x_max)
    line_diff.set_data(x_vals, sigma_vals)
    title_diff.set_text(rf"Diffusion $\hat\sigma_\theta(t,x)$ at $t={t0:.2f}$")
    return line_diff, title_diff

anim_diff = FuncAnimation(
    fig_diff,
    update_diff,
    init_func=init_diff,
    frames=len(t_frames),
    interval=80,
    blit=True,
)

plt.close(fig_diff)
display(HTML(anim_diff.to_jshtml()))


## Surrogate apply helper & Data Generation

In [ ]:
# Helper: evaluate learned SDE surrogate at (t,x) + Generate surrogate paths for Stage-2 data
# Requires: params_sde, in_norm_sde, f_sigma_hat, cfg, key_main (optional)

import math
from dataclasses import dataclass
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

def surrogate_f_sigma(params_sde, in_norm_sde, t, x, activation="tanh"):
    """
    Evaluate the learned SDE surrogate (f̂, σ̂) at (t, x).

    Inputs:
      t : scalar, array, or broadcastable with x
      x : scalar, array, or broadcastable with t
    Returns:
      f_hat      : array with broadcasted shape
      sigma_hat  : array with broadcasted shape
    """
    t_arr = jnp.asarray(t, dtype=jnp.float64)
    x_arr = jnp.asarray(x, dtype=jnp.float64)

    t_b, x_b = jnp.broadcast_arrays(t_arr, x_arr)
    tx = jnp.stack([t_b.ravel(), x_b.ravel()], axis=-1)  # (B,2)

    tx_norm = in_norm_sde(tx)

    f_hat_flat, sigma_hat_flat = f_sigma_hat(
        params_sde,
        tx_norm,
        activation=activation,
        sigma_min=cfg.sigma_min,
    )

    f_hat = f_hat_flat.reshape(t_b.shape)
    sigma_hat = sigma_hat_flat.reshape(t_b.shape)
    return f_hat, sigma_hat

# -------------------------------------------------------------------
# Generate data from the learned neural SDE surrogate for Stage-2 training
# -------------------------------------------------------------------
@dataclass
class CFGGen:
    T: float = float(getattr(cfg, "T", 5.0))
    dt: float = float(getattr(cfg, "dt", 0.01))
    n_traj: int = 256

    # Trusted x-domain (set to match surrogate training support)
    x_min_train: float = -2.0
    x_max_train: float =  2.0

    # Diversify initial conditions
    x0_mode: str = "uniform"  # "point" | "uniform" | "normal"
    x0_value: float = 0.0
    x0_mean: float = 0.0
    x0_std: float = 0.5
    x0_low: float = -2.0
    x0_high: float = 2.0

    # ---------------- Stage-2 TX_gen mixture knobs ----------------
    tx_mix_uniform_frac: float = 0.7   # fraction of TX_gen sampled uniformly in-domain
    tx_mix_seed: int = 0               # seed for uniform sampling + subsampling

    # ---------------- Trajectory diversity selection (NEW) ----------------
    traj_balance_enable: bool = True   # turn ON/OFF the balanced-trajectory selection
    traj_balance_bins: int = 41        # number of x-bins used to measure coverage
    traj_balance_oversample: int = 12   # simulate this many times more trajs, then select best subset
    traj_balance_seed: int = 0         # deterministic selection seed (separate from tx_mix_seed)



cfg_gen = CFGGen()

def _sample_x0(key, n_traj: int, cfg_gen: CFGGen):
    """Sample initial conditions x0 with minimal knobs."""
    if cfg_gen.x0_mode == "point":
        return jnp.full((n_traj,), float(cfg_gen.x0_value), dtype=jnp.float64)

    if cfg_gen.x0_mode == "uniform":
        return jax.random.uniform(
            key,
            (n_traj,),
            minval=float(cfg_gen.x0_low),
            maxval=float(cfg_gen.x0_high),
            dtype=jnp.float64,
        )

    if cfg_gen.x0_mode == "normal":
        x0 = (
            jax.random.normal(key, (n_traj,), dtype=jnp.float64) * float(cfg_gen.x0_std)
            + float(cfg_gen.x0_mean)
        )
        return jnp.clip(x0, float(cfg_gen.x_min_train), float(cfg_gen.x_max_train))

    raise ValueError(f"Unknown cfg_gen.x0_mode = {cfg_gen.x0_mode}")

def _reflect_to_interval(x, a: float, b: float):
    """
    Reflect x into [a, b] (vectorized), i.e. mirror at boundaries instead of clipping.
    Works for values arbitrarily far outside the interval.
    """
    a = jnp.asarray(a, dtype=jnp.float64)
    b = jnp.asarray(b, dtype=jnp.float64)
    w = b - a
    # map to [0, 2w)
    y = jnp.mod(x - a, 2.0 * w)
    # reflect [w, 2w) back to [0, w]
    y_ref = jnp.where(y <= w, y, 2.0 * w - y)
    return a + y_ref

def simulate_surrogate_paths(key, params_sde, in_norm_sde, cfg_gen: CFGGen):
    """
    Euler–Maruyama on the learned surrogate:
        x_{n+1} = x_n + f̂(t_n, x_n) dt + σ̂(t_n, x_n) sqrt(dt) ξ_n

    NEW (optional):
      - Oversample candidate trajectories and SELECT a subset whose x-occupancy over time
        is closer to uniform in [x_min_train, x_max_train].
      - This changes only WHICH trajectories are kept, not the SDE simulation itself.

    Returns:
      t_sim : (N+1,)
      x_sim : (n_traj, N+1)   (always exactly cfg_gen.n_traj)
    """
    dt = float(cfg_gen.dt)
    T  = float(cfg_gen.T)
    n_traj_target = int(cfg_gen.n_traj)

    x_min_train = float(cfg_gen.x_min_train)
    x_max_train = float(cfg_gen.x_max_train)

    N = int(T / dt)
    t_sim = jnp.linspace(0.0, T, N + 1, dtype=jnp.float64)

    # ---------------- NEW: oversample + selection knobs ----------------
    balance_on = bool(getattr(cfg_gen, "traj_balance_enable", False))
    oversample = int(getattr(cfg_gen, "traj_balance_oversample", 1))
    oversample = max(1, oversample)
    n_traj_sim = n_traj_target * oversample if balance_on else n_traj_target

    # diversified initial condition (kept in-domain)
    key_x0, key_noise = jax.random.split(key, 2)

    # If balancing is ON, force x0 to be stratified across the interval (reduces bias immediately)
    if balance_on:
        # stratified x0: one per stratum, shuffled
        key_perm, key_jit = jax.random.split(key_x0, 2)
        u = (jnp.arange(n_traj_sim, dtype=jnp.float64) + 0.5) / float(n_traj_sim)  # midpoints
        x0 = x_min_train + (x_max_train - x_min_train) * u
        x0 = x0 + (jax.random.uniform(key_jit, (n_traj_sim,), dtype=jnp.float64) - 0.5) * (
            (x_max_train - x_min_train) / float(n_traj_sim)
        )
        perm = jax.random.permutation(key_perm, n_traj_sim)
        x0 = x0[perm]
    else:
        x0 = _sample_x0(key_x0, n_traj_sim, cfg_gen)

    x0 = jnp.clip(x0, x_min_train, x_max_train)

    dW = jax.random.normal(key_noise, (N, n_traj_sim), dtype=jnp.float64) * math.sqrt(dt)

    def step(x_t, inputs):
        k_idx, t_curr = inputs
        t_vec = jnp.full_like(x_t, t_curr)

        # keep surrogate evaluation in-domain
        x_t_clip = jnp.clip(x_t, x_min_train, x_max_train)

        f_hat, sigma_hat = surrogate_f_sigma(
            params_sde, in_norm_sde, t_vec, x_t_clip, activation="tanh"
        )
        x_next = x_t + f_hat * dt + sigma_hat * dW[k_idx]

        # REFLECT (instead of clip) so the marginal doesn't pile up at the walls
        x_next = _reflect_to_interval(x_next, x_min_train, x_max_train)
        return x_next, x_next

    idxs = jnp.arange(N, dtype=jnp.int32)
    _, x_hist = jax.lax.scan(step, x0, (idxs, t_sim[:-1]))  # (N, n_traj_sim)
    x_sim_all = jnp.concatenate([x0[None, :], x_hist], axis=0).T  # (n_traj_sim, N+1)

    # ---------------- NEW: select subset of trajectories for balanced x-coverage ----------------
    if balance_on and (n_traj_sim > n_traj_target):
        bins = int(getattr(cfg_gen, "traj_balance_bins", 41))
        bins = max(5, bins)

        # Use numpy for selection (not jitted; this function is called eagerly anyway)
        x_np = np.asarray(x_sim_all, dtype=np.float64)  # (n_traj_sim, N+1)

        # Per-trajectory x-occupancy histograms (counts over time)
        edges = np.linspace(x_min_train, x_max_train, bins + 1)
        H = np.zeros((n_traj_sim, bins), dtype=np.float64)
        for i in range(n_traj_sim):
            H[i], _ = np.histogram(x_np[i], bins=edges)

        # Greedy selection to match uniform target histogram
        # target = uniform occupancy over x-bins after selecting n_traj_target trajectories
        total_points = float(n_traj_target) * float(x_np.shape[1])  # x_np.shape[1] = N+1
        target = np.full((bins,), total_points / float(bins), dtype=np.float64)

        rng_sel = np.random.default_rng(int(getattr(cfg_gen, "traj_balance_seed", 0)))
        order = rng_sel.permutation(n_traj_sim)

        chosen = []
        cur = np.zeros((bins,), dtype=np.float64)

        # score = squared error to target
        def score(v):
            d = v - target
            return float(d @ d)

        # Greedy: add the trajectory that most decreases the score
        # (in permuted order tie-breaks deterministically)
        available = list(order)
        for _ in range(n_traj_target):
            best_j = None
            best_sc = None
            # small optimization: check a chunk instead of all, for speed
            for j in available:
                sc = score(cur + H[j])
                if (best_sc is None) or (sc < best_sc):
                    best_sc = sc
                    best_j = j
            chosen.append(best_j)
            cur = cur + H[best_j]
            available.remove(best_j)

        chosen = np.array(chosen, dtype=np.int64)
        x_sim = jnp.asarray(x_np[chosen], dtype=jnp.float64)
    else:
        x_sim = x_sim_all

    return t_sim, x_sim

# Simulate surrogate paths
if "key_main" in globals():
    key_main, key_surr = jax.random.split(key_main)
else:
    key_surr = jax.random.PRNGKey(2026)

t_surr, x_surr = simulate_surrogate_paths(key_surr, params_sde, in_norm_sde, cfg_gen)

print("Surrogate sim shapes: t_surr =", t_surr.shape, ", x_surr =", x_surr.shape)
print("x_surr range:", float(x_surr.min()), float(x_surr.max()))

def build_tx_dataset(t, x):
    """
    Flatten paths into a cloud of (t_n, x_n) points:
      t : (N+1,)
      x : (n_traj, N+1)
    Returns:
      TX_gen : (B, 2) where B = n_traj * (N+1)
    """
    t_b = jnp.broadcast_to(t[None, :], x.shape)
    t_flat = t_b.reshape(-1, 1)
    x_flat = x.reshape(-1, 1)
    return jnp.concatenate([t_flat, x_flat], axis=1)

# ---------------------------
# Build Stage-2 training cloud TX_gen as a MIXTURE:
#   (1) trajectory points from surrogate paths
#   (2) uniform random points over [0,T] x [x_min_train, x_max_train]
# Mixture proportion controlled by cfg_gen.tx_mix_uniform_frac
# ---------------------------

TX_traj = build_tx_dataset(t_surr, x_surr)   # (B_traj, 2)
B_traj = int(TX_traj.shape[0])

# How many uniform points?
u_frac = float(getattr(cfg_gen, "tx_mix_uniform_frac", 0.0))
u_frac = float(np.clip(u_frac, 0.0, 1.0))
B_uni = int(np.round(u_frac * B_traj))
B_trj_keep = B_traj - B_uni

# RNG for mixing (numpy RNG is fine here; outputs get converted to jnp)
rng_mix = np.random.default_rng(int(getattr(cfg_gen, "tx_mix_seed", 0)))

# --- (2) uniform rectangle points ---
if B_uni > 0:
    t_lo = 0.0
    t_hi = float(cfg_gen.T)
    x_lo = float(cfg_gen.x_min_train)
    x_hi = float(cfg_gen.x_max_train)

    t_uni = rng_mix.uniform(t_lo, t_hi, size=B_uni)
    x_uni = rng_mix.uniform(x_lo, x_hi, size=B_uni)
    TX_uni = np.stack([t_uni, x_uni], axis=1).astype(np.float64)  # (B_uni, 2)
else:
    TX_uni = np.zeros((0, 2), dtype=np.float64)

# --- (1) trajectory points: optionally SUBSAMPLE to keep total size fixed ---
TX_traj_np = np.asarray(TX_traj, dtype=np.float64)

if B_trj_keep < B_traj:
    idx = rng_mix.choice(B_traj, size=B_trj_keep, replace=False)
    TX_traj_np = TX_traj_np[idx]
# else: keep all trajectory points

# Concatenate + shuffle (so batches don't come in two blocks)
TX_gen_np = np.concatenate([TX_traj_np, TX_uni], axis=0)
perm = rng_mix.permutation(TX_gen_np.shape[0])
TX_gen_np = TX_gen_np[perm]

TX_gen = jnp.asarray(TX_gen_np, dtype=jnp.float64)
print("TX_gen shape:", TX_gen.shape, f"(traj_keep={TX_traj_np.shape[0]}, uni={TX_uni.shape[0]}, u_frac={u_frac:.3f})")

# Keep the filter too (harmless redundancy; also protects against any future changes)
x_min_train = float(cfg_gen.x_min_train)
x_max_train = float(cfg_gen.x_max_train)

TX_gen_np = np.asarray(TX_gen)
mask = (TX_gen_np[:, 1] >= x_min_train) & (TX_gen_np[:, 1] <= x_max_train)
TX_gen_np = TX_gen_np[mask]
TX_gen = jnp.asarray(TX_gen_np, dtype=jnp.float64)

print("TX_gen filtered shape:", TX_gen.shape)

globals().update({
    "t_surr": t_surr,
    "x_surr": x_surr,
    "TX_gen": TX_gen,
    "cfg_gen": cfg_gen,
})
print("Exported: t_surr, x_surr, TX_gen, cfg_gen")

# Quick plot: sample a few surrogate paths (NOTE: now reflected into [-2,2])
n_plot = min(20, x_surr.shape[0])
idx_plot = np.linspace(0, x_surr.shape[0] - 1, n_plot, dtype=int)

plt.figure(figsize=(6, 4))
for idx in idx_plot:
    plt.plot(np.asarray(t_surr), np.asarray(x_surr[idx]), alpha=0.6)
# --- overlay a small subsample of TX_gen to verify uniform coverage ---
TX_plot_np = np.asarray(TX_gen)
rng_vis = np.random.default_rng(0)
M = min(5000, TX_plot_np.shape[0])  # cap so the plot stays fast
idx_vis = rng_vis.choice(TX_plot_np.shape[0], size=M, replace=False)

plt.scatter(
    TX_plot_np[idx_vis, 0],
    TX_plot_np[idx_vis, 1],
    s=3,
    alpha=0.15,
)

plt.xlabel("t")
plt.ylabel("x_t (surrogate paths)")
plt.title("Sample trajectories from neural SDE surrogate (reflect + diverse x0)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Generator Neural Net Setup

In [ ]:
# Generator neural net setup: τ_i(t) and ξ_i(t,x) for SDE symmetries
# Requires:
#   - jax, jax.numpy as jnp
#   - init_mlp_params, mlp_forward
#   - Normalizer, fit_normalizer
#   - TX_gen (flattened (t,x) pairs from surrogate simulation)
#   - key_main (PRNG key)
#   - cfg (for dtype / consistency)

from dataclasses import dataclass
import jax
import jax.numpy as jnp

# ----------------- Config for generator networks ------------------------------

from dataclasses import dataclass

@dataclass
class GenConfig:
    n_generators: int = 3
    hidden_tau: int = 128
    hidden_xi: int = 128
    depth_tau: int = 3     # NEW: number of hidden layers
    depth_xi: int = 3      # NEW
    activation: str = "swish"   # NEW: swish (SiLU)

gen_cfg = GenConfig()



gen_cfg = GenConfig()

# ----------------- Normalizers for generator inputs ---------------------------

# TX_gen has shape (B, 2) with columns [t, x]
t_flat_gen = TX_gen[:, 0:1]   # (B,1)
# We could also normalize x separately if desired, but a joint (t,x) norm is handy.
tx_norm_gen = fit_normalizer(TX_gen)       # for ξ(t,x)
t_norm_gen = fit_normalizer(t_flat_gen)    # for τ(t)

print("Gen t_norm mean/std:", t_norm_gen.mean, t_norm_gen.std)
print("Gen tx_norm mean/std:", tx_norm_gen.mean, tx_norm_gen.std)

# ----------------- Per-generator network builders -----------------------------

def tau_forward(params, t_norm, activation="tanh"):
    """
    Forward pass for τ(t).
    t_norm: (..., 1) normalized time.
    """
    return mlp_forward(params, t_norm, activation=gen_cfg.activation)[..., 0:1]

def xi_forward(params, tx_norm, activation="tanh"):
    """
    Forward pass for ξ(t,x).
    tx_norm: (..., 2) normalized (t,x).
    """
    return mlp_forward(params, tx_norm, activation=gen_cfg.activation)[..., 0:1]

def init_generator_params(key, gen_cfg: GenConfig):
    """
    Initialize parameters for all generators.
    Returns:
      params_gen = {
        "tau":  [params_tau_i  for i in range(m)],
        "xi":   [params_xi_i   for i in range(m)],
      }
    """
    m = gen_cfg.n_generators
    keys = jax.random.split(key, 2 * m)

    params_tau_list = []
    params_xi_list = []

    for i in range(m):
        k_tau  = keys[2 * i]
        k_xi   = keys[2 * i + 1]

        params_tau = init_mlp_params(
            k_tau,
            sizes=[1, gen_cfg.hidden_tau, gen_cfg.hidden_tau, 1],
        )

        params_xi = init_mlp_params(
            k_xi,
            sizes=[2, gen_cfg.hidden_xi, gen_cfg.hidden_xi, 1],
        )

        params_tau_list.append(params_tau)
        params_xi_list.append(params_xi)

    return {"tau": params_tau_list, "xi": params_xi_list}


# Initialize generator parameters
key_main, key_gen = jax.random.split(key_main)
params_gen = init_generator_params(key_gen, gen_cfg)

# ----------------- Convenience evaluator for all generators -------------------

def eval_generators(params_gen, t, x, activation=None):
    """
    Evaluate all generators at (t,x).
    Returns:
      tau_vals : (m, ...) τ_i(t)
      xi_vals  : (m, ...) ξ_i(t,x)
    """
    if activation is None:
        activation = gen_cfg.activation

    t_arr = jnp.asarray(t, dtype=jnp.float64)
    x_arr = jnp.asarray(x, dtype=jnp.float64)

    t_b, x_b = jnp.broadcast_arrays(t_arr, x_arr)
    B_shape = t_b.shape

    t_flat = t_b.reshape(-1, 1)
    x_flat = x_b.reshape(-1, 1)
    tx_flat = jnp.concatenate([t_flat, x_flat], axis=1)

    t_norm  = t_norm_gen(t_flat)
    tx_norm = tx_norm_gen(tx_flat)

    tau_list, xi_list = [], []

    for params_tau, params_xi in zip(params_gen["tau"], params_gen["xi"]):
        tau_flat = tau_forward(params_tau, t_norm, activation=activation)
        xi_flat  = xi_forward(params_xi,  tx_norm, activation=activation)
        tau_list.append(tau_flat.reshape(B_shape))
        xi_list.append(xi_flat.reshape(B_shape))

    tau_vals = jnp.stack(tau_list, axis=0)  # (m, ...)
    xi_vals  = jnp.stack(xi_list,  axis=0)  # (m, ...)
    return tau_vals, xi_vals


# JIT-ed version for speed if desired
eval_generators_jit = jax.jit(eval_generators, static_argnames=("activation",))


print(f"Initialized generator nets with m = {gen_cfg.n_generators} generators.")

# Export to globals for later stages (invariance loss, Lie algebra loss, etc.)
globals().update({
    "gen_cfg": gen_cfg,
    "params_gen": params_gen,
    "t_norm_gen": t_norm_gen,
    "tx_norm_gen": tx_norm_gen,
    "tau_forward": tau_forward,
    "xi_forward": xi_forward,
    "eval_generators": eval_generators,
    "eval_generators_jit": eval_generators_jit,
})




In [ ]:
import jax
import jax.numpy as jnp

def eval_generators_tau_xi(params_gen, t, x, *, activation="tanh", normalize_tx=None):
    """
    SDE generator evaluator: returns only (tau, xi).

    Inputs:
      t, x: (B,) arrays
    Returns:
      tau_all: (m, B)
      xi_all:  (m, B)
    """
    t = jnp.asarray(t, dtype=jnp.float64)
    x = jnp.asarray(x, dtype=jnp.float64)
    if t.ndim != 1 or x.ndim != 1:
        raise ValueError("eval_generators_tau_xi expects t and x as 1D arrays of shape (B,)")

    # Optional preprocessing in raw (t,x) space
    if normalize_tx is None:
        t_raw, x_raw = t, x
    else:
        t_raw, x_raw = normalize_tx(t, x)
        t_raw = jnp.asarray(t_raw, dtype=jnp.float64)
        x_raw = jnp.asarray(x_raw, dtype=jnp.float64)

    # Build (B,1) and (B,2) inputs
    t_col  = t_raw.reshape(-1, 1)                         # (B,1)
    x_col  = x_raw.reshape(-1, 1)                         # (B,1)
    tx_col = jnp.concatenate([t_col, x_col], axis=1)      # (B,2)

    # Apply the SAME normalizers used in generator training
    t_norm  = t_norm_gen(t_col)       # (B,1)
    tx_norm = tx_norm_gen(tx_col)     # (B,2)

    taus = []
    xis  = []

    for params_tau, params_xi in zip(params_gen["tau"], params_gen["xi"]):
        tau_flat = tau_forward(params_tau, t_norm, activation=activation).reshape(-1)  # (B,)
        xi_flat  = xi_forward(params_xi,  tx_norm, activation=activation).reshape(-1) # (B,)
        taus.append(tau_flat)
        xis.append(xi_flat)

    tau_all = jnp.stack(taus, axis=0)  # (m,B)
    xi_all  = jnp.stack(xis,  axis=0)  # (m,B)
    return tau_all, xi_all

# JIT wrapper (recommended)
eval_generators_tau_xi_jit = jax.jit(eval_generators_tau_xi, static_argnames=("activation", "normalize_tx"))


# Algebraic Losses

## Loss 1

In [ ]:
# ============================ S1 — Lie bracket closure + constancy (fixed version) ============================
# Compatible with:
#   - params_gen: {"tau": [params_tau_i], "xi": [params_xi_i]}
#   - gen_cfg.n_generators
#   - t_norm_gen, tx_norm_gen
#   - tau_forward, xi_forward
#
# Vector fields: X_i = τ_i(t) ∂_t + ξ_i(t,x) ∂_x.
# We enforce:
#   - [X_i, X_j] stays in span{X_k} via projection
#   - structure coefficients c_ij^k approximately constant over (t,x).

import jax
import jax.numpy as jnp

def _ordered_pair_indices(n: int):
    """
    Return (i,j) pairs with i != j, in a deterministic order.
    Used to enumerate all brackets [X_i, X_j].
    """
    idx = jnp.arange(n, dtype=jnp.int32)
    ii  = jnp.repeat(idx, repeats=n-1)
    base = jnp.arange(n - 1, dtype=jnp.int32)
    i_col = idx[:, None]
    jj_mat = base + (base >= i_col).astype(jnp.int32)
    jj = jj_mat.reshape(-1)
    return ii, jj

def make_s1_lie_loss(n_generators: int, rcond: float = 1e-6):
    """
    Constructs the S1 Lie algebra loss:
      - Closure: project [X_i, X_j] onto span{X_k} at each (t,x)
                 and penalize the projection error.
      - Constancy: penalize variation of the (pointwise) structure
                   coefficients c_ij^k over (t,x).

    Args:
      n_generators: number of learned generators m.
      rcond: small regularization parameter for Gram matrix inversion.

    Returns:
      loss_fn(params_gen, tx_batch) -> (scalar_loss, aux_dict)
    """
    idx_i, idx_j = _ordered_pair_indices(n_generators)
    K = int(idx_i.shape[0])
    reg = jnp.asarray(rcond, dtype=jnp.float64) ** 2

    # ---------- helper: scalar forwards for τ_i and ξ_i ----------------------

    def _tau_val_and_dt(params_tau_i, t_scalar):
        """τ_i(t) and ∂_t τ_i(t) for a single generator."""
        def tau_scalar(tt):
            t_arr = jnp.asarray([[tt]], dtype=jnp.float64)    # (1,1)
            t_norm = t_norm_gen(t_arr)                        # (1,1)
            out = tau_forward(params_tau_i, t_norm,
                              activation=gen_cfg.activation)
            return out[0, 0]  # scalar
        tau_val = tau_scalar(t_scalar)
        tau_t = jax.grad(tau_scalar)(t_scalar)
        return tau_val, tau_t

    def _xi_val_and_derivs(params_xi_i, t_scalar, x_scalar):
        """ξ_i(t,x), ξ_t, ξ_x for a single generator at (t,x)."""
        def xi_t_fun(tt):
            tx_arr = jnp.asarray([[tt, x_scalar]], dtype=jnp.float64)
            tx_norm = tx_norm_gen(tx_arr)
            out = xi_forward(params_xi_i, tx_norm,
                             activation=gen_cfg.activation)
            return out[0, 0]

        def xi_x_fun(xx):
            tx_arr = jnp.asarray([[t_scalar, xx]], dtype=jnp.float64)
            tx_norm = tx_norm_gen(tx_arr)
            out = xi_forward(params_xi_i, tx_norm,
                             activation=gen_cfg.activation)
            return out[0, 0]

        xi_val = xi_t_fun(t_scalar)
        xi_t = jax.grad(xi_t_fun)(t_scalar)
        xi_x = jax.grad(xi_x_fun)(x_scalar)
        return xi_val, xi_t, xi_x

    # ---------- helper: fields and derivatives at a single (t,x) -------------

    def _fields_and_derivs_at_point(params_gen, t_scalar, x_scalar):
        """
        Compute τ_i, ξ_i and their needed derivatives at (t,x) for all generators.
        Returns:
          tau   : (m,)
          xi    : (m,)
          tau_t : (m,)
          xi_t  : (m,)
          xi_x  : (m,)
        """
        tau_params = params_gen["tau"]
        xi_params  = params_gen["xi"]

        tau_list   = []
        xi_list    = []
        tau_t_list = []
        xi_t_list  = []
        xi_x_list  = []

        for p_tau, p_xi in zip(tau_params, xi_params):
            tau_i, tau_t_i = _tau_val_and_dt(p_tau, t_scalar)
            xi_i, xi_t_i, xi_x_i = _xi_val_and_derivs(p_xi, t_scalar, x_scalar)

            tau_list.append(tau_i)
            xi_list.append(xi_i)
            tau_t_list.append(tau_t_i)
            xi_t_list.append(xi_t_i)
            xi_x_list.append(xi_x_i)

        tau   = jnp.stack(tau_list, axis=0)    # (m,)
        xi    = jnp.stack(xi_list, axis=0)     # (m,)
        tau_t = jnp.stack(tau_t_list, axis=0)  # (m,)
        xi_t  = jnp.stack(xi_t_list, axis=0)   # (m,)
        xi_x  = jnp.stack(xi_x_list, axis=0)   # (m,)

        return tau, xi, tau_t, xi_t, xi_x

    # ---------- helper: bracket closure + coefficients at a point ------------

    def _point_err_and_C(tau, xi, tau_t, xi_t, xi_x):
        """
        Single-point closure + structure-coefficient computation.

        X_i = τ_i ∂_t + ξ_i ∂_x

        Lie bracket components:
          [X_i, X_j]^t = τ_i τ_{j,t} - τ_j τ_{i,t}
          [X_i, X_j]^x = τ_i ξ_{j,t} + ξ_i ξ_{j,x}
                         - τ_j ξ_{i,t} - ξ_j ξ_{i,x}
        """
        # V: (2,m) with rows [τ; ξ]
        V = jnp.stack([tau, xi], axis=0)  # (2, m)

        # Slice i,j components
        tau_i, tau_j = tau[idx_i], tau[idx_j]       # (K,)
        xi_i,  xi_j  = xi[idx_i],  xi[idx_j]        # (K,)
        tau_t_i, tau_t_j = tau_t[idx_i], tau_t[idx_j]
        xi_t_i,  xi_t_j  = xi_t[idx_i],  xi_t[idx_j]
        xi_x_i,  xi_x_j  = xi_x[idx_i],  xi_x[idx_j]

        # Bracket components for all ordered pairs (i,j)
        a = tau_i * tau_t_j - tau_j * tau_t_i
        b = (
            tau_i * xi_t_j + xi_i * xi_x_j
            - tau_j * xi_t_i - xi_j * xi_x_i
        )

        B = jnp.stack([a, b], axis=0)  # (2, K)

        # Project B onto span(V)
        G = V @ V.T                               # (2,2)
        G_reg = G + reg * jnp.eye(2, dtype=G.dtype)
        X = jnp.linalg.solve(G_reg, B)           # (2,K)
        C = V.T @ X                              # (m,K)
        P_B = V @ C                              # (2,K)
        E = B - P_B                              # (2,K)

        err = jnp.sum(jnp.abs(E))
        return err, C

    # ---------- main loss over batch -----------------------------------------

    def _loss_impl(params_gen, tx_batch: jnp.ndarray):
        """
        tx_batch: (B,2) with columns [t, x].
        """
        def eval_at_z(z):
            t_z, x_z = z[0], z[1]
            return _fields_and_derivs_at_point(params_gen, t_z, x_z)

        taus, xis, tau_ts, xi_ts, xi_xs = jax.vmap(eval_at_z)(tx_batch)
        # shapes: each (B, m)

        errs, Cs = jax.vmap(_point_err_and_C)(taus, xis, tau_ts, xi_ts, xi_xs)
        # errs: (B,)
        # Cs:   (B, m, K)

        error_sum = jnp.sum(errs)

        # Constancy across batch: variance of C over (t,x)
        C_var = jnp.var(Cs, axis=0)  # (m, K)
        var_sum = jnp.sum(C_var)

        total = error_sum + var_sum
        aux = {
            "error_sum": error_sum,
            "var_sum": var_sum,
        }
        return total, aux

    return jax.jit(_loss_impl)


## Loss 2 - Jacobi Identity

In [ ]:
# ============================ S2 — Jacobi identity (nested brackets, fixed) ============================
# Uses per-generator Xi_i(t,x) with its own (2x2) Jacobian and (2x2x2) Hessian,
# so that all matrix–vector products are dimensionally consistent.

import jax
import jax.numpy as jnp

def make_s2_jacobi_loss_nested(n_generators: int):
    # All distinct index triples i < j < k
    triples = [
        (i, j, k)
        for i in range(n_generators)
        for j in range(i + 1, n_generators)
        for k in range(j + 1, n_generators)
    ]
    if not triples:
        def _zero(params_gen, tx_batch):
            return jnp.array(0.0, dtype=jnp.float64), {
                "per_point": jnp.zeros((tx_batch.shape[0],), dtype=jnp.float64),
                "num_triples": 0,
            }
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)

    # All 6 permutations for symmetrized Jacobi expression
    perms6 = jnp.array(
        [[0, 1, 2],
         [0, 2, 1],
         [1, 0, 2],
         [1, 2, 0],
         [2, 0, 1],
         [2, 1, 0]],
        dtype=jnp.int32,
    )

    # -------- helper: F, J, H at a single (t,x) via per-generator Xi_i -------

    def _fields_jac_hess(params_gen, z):
        """
        Compute:
          F: (m,2)      vector field values at z = (t,x)
          J: (m,2,2)    Jacobians D X_i(z)
          H: (m,2,2,2)  Hessians D^2 X_i(z)
        for all generators X_i.
        """
        t_z, x_z = z[0], z[1]

        F_list = []
        J_list = []
        H_list = []

        # Iterate over generators; loop size m is static so jit-friendly.
        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):

            def Xi(zz):
                tt, xx = zz[0], zz[1]

                t_arr = jnp.asarray([[tt]], dtype=jnp.float64)          # (1,1)
                tx_arr = jnp.asarray([[tt, xx]], dtype=jnp.float64)     # (1,2)

                t_norm = t_norm_gen(t_arr)      # (1,1)
                tx_norm = tx_norm_gen(tx_arr)   # (1,2)

                tau_val = tau_forward(
                    params_tau_i,
                    t_norm,
                    activation=gen_cfg.activation,
                )[0, 0]  # scalar

                xi_val = xi_forward(
                    params_xi_i,
                    tx_norm,
                    activation=gen_cfg.activation,
                )[0, 0]   # scalar

                return jnp.array([tau_val, xi_val], dtype=jnp.float64)  # (2,)

            # Value, Jacobian, Hessian for generator i
            Fi = Xi(z)                                                # (2,)
            Ji = jax.jacobian(Xi)(z)                                  # (2,2)
            Hi = jax.jacobian(lambda zz: jax.jacobian(Xi)(zz))(z)     # (2,2,2)

            F_list.append(Fi)
            J_list.append(Ji)
            H_list.append(Hi)

        F = jnp.stack(F_list, axis=0)   # (m,2)
        J = jnp.stack(J_list, axis=0)   # (m,2,2)
        H = jnp.stack(H_list, axis=0)   # (m,2,2,2)

        return F, J, H

    # ------------- helper: basic bracket and nested bracket algebra ----------

    def _bracket_val(F, J, p, q):
        """
        [X_p, X_q](z) = J_q(z) @ F_p(z) - J_p(z) @ F_q(z)
        where F_i ∈ R^2, J_i ∈ R^{2x2}.
        Returns a 2-vector (in (∂_t, ∂_x) basis).
        """
        Jp, Jq = J[p], J[q]   # (2,2)
        fp, fq = F[p], F[q]   # (2,)
        return (Jq @ fp) - (Jp @ fq)   # (2,)

    def _dir_along(F, J, H, r, p, q):
        """
        Directional action of [X_p, X_q] on X_r, using first and second derivatives.
        Matches original nested-bracket algebra, specialized to (t,x).
        """
        Jr, Jp, Jq = J[r], J[p], J[q]     # (2,2)
        Hp, Hq = H[p], H[q]               # (2,2,2)
        fr, fp, fq = F[r], F[p], F[q]     # (2,)

        # (2,) results
        t1 = Jq @ (Jp @ fr)
        t2 = ((Hq * fr[None, None, :]).sum(axis=2)) @ fp
        t3 = Jp @ (Jq @ fr)
        t4 = ((Hp * fr[None, None, :]).sum(axis=2)) @ fq

        return t1 + t2 - t3 - t4

    def _double_bracket(F, J, H, r, p, q):
        """
        Nested bracket [[X_r, X_p], X_q] at a point.
        """
        inner = _bracket_val(F, J, p, q)
        return _dir_along(F, J, H, r, p, q) - (J[r] @ inner)

    def _jacobi_one_order(F, J, H, u, v, w):
        """
        Jacobi combination for one ordering (u, v, w):
          [[X_u, X_v], X_w] + [[X_w, X_u], X_v] + [[X_v, X_w], X_u]
        """
        return (
            _double_bracket(F, J, H, u, v, w)
            + _double_bracket(F, J, H, w, u, v)
            + _double_bracket(F, J, H, v, w, u)
        )

    def _triple_sum_over_6(F, J, H, i, j, k):
        """
        Symmetrize over all 6 permutations of (i,j,k), summing absolute values.
        """
        inds = jnp.array([i, j, k], dtype=jnp.int32)

        def _one_perm(p):
            u, v, w = inds[p[0]], inds[p[1]], inds[p[2]]
            r = _jacobi_one_order(F, J, H, u, v, w)  # (2,)
            return jnp.sum(jnp.abs(r))

        vals = jax.vmap(_one_perm)(perms6)  # (6,)
        return jnp.sum(vals)

    def _point_loss(params_gen, z):
        """
        Jacobi loss at a single point z = (t,x), summed over all triples (i,j,k).
        """
        F, J, H = _fields_jac_hess(params_gen, z)
        per_tr = jax.vmap(
            lambda a, b, c: _triple_sum_over_6(F, J, H, a, b, c)
        )(tri_i, tri_j, tri_k)  # (num_triples,)
        return jnp.sum(per_tr)

    _point_loss_jit = jax.jit(_point_loss)

    def _loss_impl(params_gen, tx_batch: jnp.ndarray):
        """
        tx_batch: (B,2) array with columns [t, x].
        Returns:
          total_loss, {
              "per_point": (B,),
              "num_triples": int,
          }
        """
        per_point = jax.vmap(lambda z: _point_loss_jit(params_gen, z))(tx_batch)
        total = jnp.sum(per_point)
        aux = {
            "per_point": per_point,
            "num_triples": int(tri_i.shape[0]),
        }
        return total, aux

    return jax.jit(_loss_impl)


## Loss 3 - Skewsymmetry

In [ ]:
# ============================ S3 — Skew-symmetry (fixed) ============================
# Uses per-generator Xi_i(t,x) so that F and J have shapes:
#   F: (m,2),  J: (m,2,2)
# and [X_i, X_j](z) = J_j @ F_i - J_i @ F_j is always well-typed.

import jax
import jax.numpy as jnp

def make_s3_skewsym_loss(n_generators: int):
    # All distinct pairs i < j
    pairs = [
        (i, j)
        for i in range(n_generators)
        for j in range(i + 1, n_generators)
    ]
    if not pairs:
        def _zero(params_gen, tx_batch):
            return jnp.array(0.0, dtype=jnp.float64), {
                "per_point": jnp.zeros((tx_batch.shape[0],), dtype=jnp.float64),
                "num_pairs": 0,
            }
        return jax.jit(_zero)

    pi = jnp.array([p[0] for p in pairs], dtype=jnp.int32)
    pj = jnp.array([p[1] for p in pairs], dtype=jnp.int32)

    # ---------- helper: F and J at a single (t,x) ----------------------------

    def _fields_and_jac(params_gen, z):
        """
        Compute:
          F: (m,2)  vector field values at z = (t,x)
          J: (m,2,2) Jacobians w.r.t (t,x)
        for all generators X_i.
        """
        t_z, x_z = z[0], z[1]

        F_list = []
        J_list = []

        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):

            def Xi(zz):
                tt, xx = zz[0], zz[1]

                t_arr = jnp.asarray([[tt]], dtype=jnp.float64)          # (1,1)
                tx_arr = jnp.asarray([[tt, xx]], dtype=jnp.float64)     # (1,2)

                t_norm = t_norm_gen(t_arr)      # (1,1)
                tx_norm = tx_norm_gen(tx_arr)   # (1,2)

                tau_val = tau_forward(
                    params_tau_i,
                    t_norm,
                    activation=gen_cfg.activation,
                )[0, 0]
                xi_val = xi_forward(
                    params_xi_i,
                    tx_norm,
                    activation=gen_cfg.activation,
                )[0, 0]

                return jnp.array([tau_val, xi_val], dtype=jnp.float64)  # (2,)

            Fi = Xi(z)                         # (2,)
            Ji = jax.jacobian(Xi)(z)          # (2,2)

            F_list.append(Fi)
            J_list.append(Ji)

        F = jnp.stack(F_list, axis=0)  # (m,2)
        J = jnp.stack(J_list, axis=0)  # (m,2,2)

        return F, J

    # ---------- bracket at a point ------------------------------------------

    def _bracket_val(F, J, p, q):
        """
        [X_p, X_q](z) = J_q(z) @ F_p(z) - J_p(z) @ F_q(z)
        """
        Jp, Jq = J[p], J[q]   # (2,2)
        fp, fq = F[p], F[q]   # (2,)
        return (Jq @ fp) - (Jp @ fq)

    def _point_loss(params_gen, z):
        """
        Skew-symmetry loss at a single point z = (t,x):
          sum_{i<j} || [X_i, X_j] + [X_j, X_i] ||_1
        """
        F, J = _fields_and_jac(params_gen, z)

        def one(i, j):
            r = _bracket_val(F, J, i, j) + _bracket_val(F, J, j, i)
            return jnp.sum(jnp.abs(r))

        vals = jax.vmap(one)(pi, pj)  # (num_pairs,)
        return jnp.sum(vals)

    _pl = jax.jit(_point_loss)

    def _loss_impl(params_gen, tx_batch: jnp.ndarray):
        """
        tx_batch: (B,2) with columns [t, x].
        Returns:
          total_loss, {
              "per_point": (B,),
              "num_pairs": int,
          }
        """
        per_point = jax.vmap(lambda z: _pl(params_gen, z))(tx_batch)
        total = jnp.sum(per_point)
        aux = {
            "per_point": per_point,
            "num_pairs": int(pi.shape[0]),
        }
        return total, aux

    return jax.jit(_loss_impl)


## Loss 4 - Bilinearity

In [ ]:
# ============================ S4 — Bilinearity (fixed) ============================
# Uses per-generator Xi_i(t,x) so F and J are well-typed:
#   F: (m,2), J: (m,2,2)
# Checks:
#   [c u + c' v, w] = c [u, w] + c' [v, w]
#   [u, c v + c' w] = c [u, v] + c' [u, w]

import jax
import jax.numpy as jnp

def make_s4_bilinearity_loss(
    n_generators: int,
    num_cc: int = 4,
    cc_list=None,
    normalize: bool = True,
):
    # All distinct triples i < j < k
    triples = [
        (i, j, k)
        for i in range(n_generators)
        for j in range(i + 1, n_generators)
        for k in range(j + 1, n_generators)
    ]
    if not triples:
        def _zero(params_gen, tx_batch, key=None):
            return jnp.array(0.0, dtype=jnp.float64), {
                "per_point": jnp.zeros((tx_batch.shape[0],), dtype=jnp.float64),
            }
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)

    perms6 = jnp.array(
        [[0, 1, 2],
         [0, 2, 1],
         [1, 0, 2],
         [1, 2, 0],
         [2, 0, 1],
         [2, 1, 0]],
        dtype=jnp.int32,
    )

    # ----------- helper: F and J at a single (t,x) ---------------------------

    def _fields_and_jac(params_gen, z):
        """
        Compute:
          F: (m,2)  vector field values at z = (t,x)
          J: (m,2,2) Jacobians w.r.t (t,x)
        for all generators X_i.
        """
        t_z, x_z = z[0], z[1]

        F_list = []
        J_list = []

        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):

            def Xi(zz):
                tt, xx = zz[0], zz[1]

                t_arr = jnp.asarray([[tt]], dtype=jnp.float64)          # (1,1)
                tx_arr = jnp.asarray([[tt, xx]], dtype=jnp.float64)     # (1,2)

                t_norm = t_norm_gen(t_arr)      # (1,1)
                tx_norm = tx_norm_gen(tx_arr)   # (1,2)

                tau_val = tau_forward(
                    params_tau_i,
                    t_norm,
                    activation=gen_cfg.activation,
                )[0, 0]
                xi_val = xi_forward(
                    params_xi_i,
                    tx_norm,
                    activation=gen_cfg.activation,
                )[0, 0]

                return jnp.array([tau_val, xi_val], dtype=jnp.float64)  # (2,)

            Fi = Xi(z)                         # (2,)
            Ji = jax.jacobian(Xi)(z)          # (2,2)

            F_list.append(Fi)
            J_list.append(Ji)

        F = jnp.stack(F_list, axis=0)  # (m,2)
        J = jnp.stack(J_list, axis=0)  # (m,2,2)

        return F, J

    # ----------- bracket and bilinearity terms at a point --------------------

    def _bracket(F, J, p, q):
        """
        [X_p, X_q](z) = J_q(z) @ F_p(z) - J_p(z) @ F_q(z)
        """
        Jp, Jq = J[p], J[q]   # (2,2)
        fp, fq = F[p], F[q]   # (2,)
        return (Jq @ fp) - (Jp @ fq)

    def _triple_terms(F, J, i, j, k, cc):
        """
        Bilinearity residuals for a single triple (i,j,k) at a fixed point,
        averaged over coefficient pairs in cc.
        """
        inds = jnp.array([i, j, k], dtype=jnp.int32)

        def one_perm(p):
            u, v, w = inds[p[0]], inds[p[1]], inds[p[2]]
            fu, fv, fw = F[u], F[v], F[w]        # (2,)
            Ju, Jv, Jw = J[u], J[v], J[w]        # (2,2)

            def one_cc(cpair):
                c, cp = cpair[0], cpair[1]

                # Linear combinations in the first slot
                f_uv = c * fu + cp * fv
                J_uv = c * Ju + cp * Jv

                # Linear combinations in the second slot
                f_vw = c * fv + cp * fw
                J_vw = c * Jv + cp * Jw

                # [c u + c' v, w]
                term1 = (Jw @ f_uv) - (J_uv @ fw)
                rhs1  = c * _bracket(F, J, u, w) + cp * _bracket(F, J, v, w)
                r1 = term1 - rhs1

                # [u, c v + c' w]
                term2 = (J_vw @ fu) - (Ju @ f_vw)
                rhs2  = c * _bracket(F, J, u, v) + cp * _bracket(F, J, u, w)
                r2 = term2 - rhs2

                if normalize:
                    denom = jnp.abs(c) + jnp.abs(cp) + 1e-12
                    r1 = r1 / denom
                    r2 = r2 / denom

                return jnp.sum(jnp.abs(r1)) + jnp.sum(jnp.abs(r2))

            vals_cc = jax.vmap(one_cc)(cc)  # (num_cc,)
            return jnp.mean(vals_cc)

        vals = jax.vmap(one_perm)(perms6)  # (6,)
        return jnp.sum(vals)

    def _point_loss(params_gen, z, cc):
        """
        Bilinearity loss at a single point z = (t,x), summed over all triples.
        """
        F, J = _fields_and_jac(params_gen, z)
        per_tr = jax.vmap(
            lambda a, b, c_: _triple_terms(F, J, a, b, c_, cc)
        )(tri_i, tri_j, tri_k)  # (num_triples,)
        return jnp.sum(per_tr)

    _pl = jax.jit(_point_loss)

    def _loss_impl(params_gen, tx_batch: jnp.ndarray, key=None):
        """
        tx_batch: (B,2) with columns [t, x].
        key: optional PRNGKey for sampling coefficient pairs if cc_list is None.

        Returns:
          total_loss, {
              "per_point": (B,),
              "num_triples": int,
              "num_cc": int,
          }
        """
        if cc_list is not None:
            cc = jnp.asarray(cc_list, dtype=jnp.float64)  # (num_cc, 2)
        else:
            key = jax.random.PRNGKey(0) if key is None else key
            cc = jax.random.uniform(
                key,
                (num_cc, 2),
                minval=-1.0,
                maxval=1.0,
                dtype=jnp.float64,
            )

        per_point = jax.vmap(lambda z: _pl(params_gen, z, cc))(tx_batch)
        total = jnp.sum(per_point)
        aux = {
            "per_point": per_point,
            "num_triples": int(tri_i.shape[0]),
            "num_cc": int(cc.shape[0]),
        }
        return total, aux

    return jax.jit(_loss_impl)


## Loss 5 - Functional Independence

In [ ]:
# ============================ PATCH: S5 loss to support eval_generators_jit returning (tau, xi, beta) ============================
import jax
import jax.numpy as jnp

def make_s5_column_independence_loss(
    n_generators: int,
    *,
    mode: str = "sigma",
    tau: float = 0.0,
    eps: float = 1e-12,
):
    # Normalize mode to a small integer code for jit-friendliness
    mode = "sigma" if mode == "sigma" else "corr_l2"
    mode_code = 0 if mode == "sigma" else 1

    def _A_from_batch(params_gen, tx_batch: jnp.ndarray):
        """
        Build A ∈ R^{2N x m} from a batch of points:
          - rows: all (τ_i(t_n), ξ_i(t_n, x_n)) stacked over n
          - columns: generators i = 0..m-1

        tx_batch: (N,2) with columns [t, x].
        """
        t_batch = tx_batch[:, 0]  # (N,)
        x_batch = tx_batch[:, 1]  # (N,)

        # eval_generators_jit now returns (tau, xi, beta) or (tau, xi, beta, phi) if return_phi=True
        outs = eval_generators_jit(params_gen, t_batch, x_batch)
        tau_vals = outs[0]  # (m, N)
        xi_vals  = outs[1]  # (m, N)

        comp = jnp.stack([tau_vals, xi_vals], axis=2)   # (m, N, 2)
        comp_B2m = jnp.transpose(comp, (1, 2, 0))       # (N, 2, m)
        A = comp_B2m.reshape(-1, n_generators)          # (2N, m)
        return A

    def _loss_impl(params_gen, tx_batch: jnp.ndarray):
        A = _A_from_batch(params_gen, tx_batch)  # (2N, m)

        # Normalize columns
        col_norms = jnp.linalg.norm(A, axis=0) + eps   # (m,)
        Ahat = A / col_norms                           # (2N, m)

        # Gram matrix of normalized columns
        G = Ahat.T @ Ahat                              # (m, m)

        if mode_code == 0:
            lam = jnp.linalg.eigvalsh(G)
            lam_min = jnp.clip(jnp.min(lam), 0.0, None)
            sigma_min = jnp.sqrt(lam_min)
            loss = jnp.maximum(
                0.0,
                jnp.asarray(tau, dtype=G.dtype) - sigma_min,
            )
            aux = {"sigma_min": sigma_min}
        else:
            I = jnp.eye(G.shape[0], dtype=G.dtype)
            off = G - I
            off = off - jnp.diag(jnp.diag(off))
            loss = jnp.sum(off * off)
            aux = {"gram_diag_mean": jnp.mean(jnp.diag(G))}

        return loss, aux

    return jax.jit(_loss_impl)

# Export patched symbol (overwrites old one)
#globals().update({"make_s5_column_independence_loss": make_s5_column_independence_loss})
#print("Patched: make_s5_column_independence_loss now supports eval_generators_jit returning (tau, xi, beta).")


# SDE Symmetry Losses

## Loss 6 - SDE Symmetry DE

In [ ]:
# ============================ S6 — SDE determining-equation loss (Gaeta–Quintero) ============================
# Implements the 1D projectable SDE symmetry determining equations:
#
#   r1 = ξ_t + f ξ_x - ξ f_x - f_t τ - f τ_t + 0.5 σ^2 ξ_xx = 0
#   r2 = σ ξ_x - ξ σ_x - τ σ_t - 0.5 σ τ_t = 0
#
# for each generator X_i = τ_i(t) ∂_t + ξ_i(t,x) ∂_x, using drift f = mu_fn(t,x),
# diffusion σ = sig_fn(t,x), and autodiff for all derivatives.

import jax
import jax.numpy as jnp

def make_s6_commutator_loss_ito(*, mu_fn, sig_fn, use_abs: bool = False):
    """
    Construct L6 loss that enforces Gaeta–Quintero SDE determining equations.

    Args:
      mu_fn(t, x):   drift function f(t,x); should be JAX-differentiable.
      sig_fn(t, x):  diffusion function σ(t,x); JAX-differentiable.
      use_abs:       if False, use squared residuals (L2); if True, use |residual| (L1).

    Returns:
      loss_fn(params_gen, tx_batch) -> (loss_scalar, aux_dict)
        - params_gen: {"tau": [...], "xi": [...]}
        - tx_batch:   (B,2) array with columns [t, x]
        - aux_dict:   {"per_point": (B,)} total residual per (t,x).
    """

    # ---- local scalar wrappers for τ(t) and ξ(t,x) with gradients ------------

    def tau_val_and_dt(params_tau, t_scalar):
        """Return τ(t), τ_t(t)."""
        def tau_scalar(tt):
            t_arr = jnp.asarray([[tt]], dtype=jnp.float64)     # (1,1)
            t_norm = t_norm_gen(t_arr)
            out = tau_forward(params_tau, t_norm,
                              activation=gen_cfg.activation)
            return out[0, 0]  # scalar
        tau_val = tau_scalar(t_scalar)
        tau_t = jax.grad(tau_scalar)(t_scalar)
        return tau_val, tau_t

    def xi_val_and_derivs(params_xi, t_scalar, x_scalar):
        """Return ξ(t,x), ξ_t, ξ_x, ξ_xx at a single (t,x)."""
        # ξ as function of t (x fixed)
        def xi_t_fun(tt):
            tx_arr = jnp.asarray([[tt, x_scalar]], dtype=jnp.float64)  # (1,2)
            tx_norm = tx_norm_gen(tx_arr)
            out = xi_forward(params_xi, tx_norm,
                             activation=gen_cfg.activation)
            return out[0, 0]

        # ξ as function of x (t fixed)
        def xi_x_fun(xx):
            tx_arr = jnp.asarray([[t_scalar, xx]], dtype=jnp.float64)  # (1,2)
            tx_norm = tx_norm_gen(tx_arr)
            out = xi_forward(params_xi, tx_norm,
                             activation=gen_cfg.activation)
            return out[0, 0]

        xi_val = xi_t_fun(t_scalar)
        xi_t = jax.grad(xi_t_fun)(t_scalar)
        xi_x = jax.grad(xi_x_fun)(x_scalar)
        xi_xx = jax.grad(lambda xx: jax.grad(xi_x_fun)(xx))(x_scalar)

        return xi_val, xi_t, xi_x, xi_xx

    # ---- drift/diffusion + their derivatives at a point ----------------------

    def f_sigma_and_derivs(t_scalar, x_scalar):
        """Compute f, f_t, f_x, σ, σ_t, σ_x at (t,x)."""
        # Drift
        def f_t_fun(tt):
            return mu_fn(tt, x_scalar)

        def f_x_fun(xx):
            return mu_fn(t_scalar, xx)

        f_val = mu_fn(t_scalar, x_scalar)
        f_t = jax.grad(f_t_fun)(t_scalar)
        f_x = jax.grad(f_x_fun)(x_scalar)

        # Diffusion
        def s_t_fun(tt):
            return sig_fn(tt, x_scalar)

        def s_x_fun(xx):
            return sig_fn(t_scalar, xx)

        sigma_val = sig_fn(t_scalar, x_scalar)
        sigma_t = jax.grad(s_t_fun)(t_scalar)
        sigma_x = jax.grad(s_x_fun)(x_scalar)

        return f_val, f_t, f_x, sigma_val, sigma_t, sigma_x

    # ---- per-point residual over all generators ------------------------------

    def _point_residual(params_gen, z):
        """
        Compute total determining-equation residual at a single point (t,x),
        summed over all generators i.
        """
        t, x = z[0], z[1]

        # SDE coefficients and their derivatives
        f_val, f_t, f_x, sigma_val, sigma_t, sigma_x = f_sigma_and_derivs(t, x)
        sigma2 = sigma_val * sigma_val

        total_res = 0.0

        for params_tau_i, params_xi_i in zip(params_gen["tau"], params_gen["xi"]):
            # Generator derivatives
            tau_i, tau_t_i = tau_val_and_dt(params_tau_i, t)
            xi_i, xi_t_i, xi_x_i, xi_xx_i = xi_val_and_derivs(params_xi_i, t, x)

            # r1 (drift determining eq)
            r1 = (
                xi_t_i
                + f_val * xi_x_i
                - xi_i * f_x
                - f_t * tau_i
                - f_val * tau_t_i
                + 0.5 * sigma2 * xi_xx_i
            )

            # r2 (diffusion determining eq)
            r2 = (
                sigma_val * xi_x_i
                - xi_i * sigma_x
                - tau_i * sigma_t
                - 0.5 * sigma_val * tau_t_i
            )

            if use_abs:
                total_res = total_res + jnp.abs(r1) + jnp.abs(r2)
            else:
                total_res = total_res + r1 * r1 + r2 * r2

        return total_res

    _point_residual_jit = jax.jit(_point_residual)

    # ---- batched loss over tx_batch -----------------------------------------

    def _loss_impl(params_gen, tx_batch: jnp.ndarray):
        """
        tx_batch: (B,2) array with columns [t, x].
        """
        per_point = jax.vmap(lambda z: _point_residual_jit(params_gen, z))(tx_batch)
        loss = jnp.mean(per_point)  # or jnp.sum(per_point); here we take mean
        aux = {"per_point": per_point}
        return loss, aux

    return jax.jit(_loss_impl)


## Loss 7 - SDE flow

In [ ]:
# ============================ S7 — Prolonged pushforward residual (SDE generators only: tau/xi) ============================
import jax
import jax.numpy as jnp

def make_s7_pushforward_coeff_loss_sde_only(
    *,
    mu_fn,
    sig_fn,
    eps: float = 1e-2,
    num_steps: int = 1,
    sigma_floor: float = 1e-8,
    dt_floor: float = 1e-10,
    dt_neg_penalty: float = 100.0,
    activation: str = "tanh",
    normalize_tx=None,
    jit: bool = True,
    # NEW safety
    tau_clip: float = 5.0,
    xi_clip: float = 5.0,
    t_clip_lo: float = -1e6,
    t_clip_hi: float =  1e6,
    x_clip_abs: float = 50.0,
):

    """
    S7 (trajectory-level) validity for SDE symmetries (tau, xi only), mirroring paper-style "after-flow residual".

    For each learned generator X_i:
      1) Integrate the prolonged epsilon-flow on (t,x,mu,sigma):
            dt/dε = τ(t)
            dx/dε = ξ(t,x)
            dσ/dε = σ(∂xξ - 1/2 ∂tτ)
            dμ/dε = ∂tξ + μ∂xξ + 1/2 σ^2 ∂xxξ - μ∂tτ
      2) Compare predicted pushed coefficients (μ_pred, σ_pred) to the surrogate evaluated at pushed points:
            μ_eval = μ(t_push, x_push),  σ_eval = σ(t_push, x_push)

    NOTE: dt-negative penalty removed intentionally (time monotonicity not enforced).
    We keep the dt_neg_penalty argument and per_gen_stats slot for compatibility (filled with zeros).
    """

    if "eval_generators_tau_xi_jit" not in globals() or (not callable(globals()["eval_generators_tau_xi_jit"])):
        raise NameError(
            "S7(SDE-only) requires a callable eval_generators_tau_xi_jit(params_gen, t, x, ...)"
        )

    eval_gen_tau_xi = globals()["eval_generators_tau_xi_jit"]

    eps = jnp.asarray(eps, dtype=jnp.float64)
    num_steps = int(num_steps)

    # -------------------- autodiff-based diagonal eval + derivatives --------------------

    def _rhs_diag_with_derivs(params_gen, t_stack, x_stack):
        """
        Returns diag-evaluations for each generator i at its own points with autodiff derivatives:
          tau, xi, tau_t, xi_t, xi_x, xi_xx   all shape (m,B)
        """
        m, B = t_stack.shape

        # Flatten points and build matching generator indices (diag pairing)
        t_flat = t_stack.reshape(-1)  # (mB,)
        x_flat = x_stack.reshape(-1)  # (mB,)
        i_flat = jnp.repeat(jnp.arange(m, dtype=jnp.int32), B)  # (mB,)

        # Scalar, clipped outputs for a specific generator i at a single (t,x)
        def _tau_i_scalar(params_gen, i, t, x):
            tau_all, _ = eval_gen_tau_xi(
                params_gen, jnp.asarray([t], dtype=jnp.float64), jnp.asarray([x], dtype=jnp.float64),
                activation=activation, normalize_tx=normalize_tx
            )
            tau = tau_all[i, 0]
            tau = jnp.nan_to_num(tau, nan=0.0, posinf=0.0, neginf=0.0)
            tau = tau_clip * jnp.tanh(tau / tau_clip)
            return tau

        def _xi_i_scalar(params_gen, i, t, x):
            _, xi_all = eval_gen_tau_xi(
                params_gen, jnp.asarray([t], dtype=jnp.float64), jnp.asarray([x], dtype=jnp.float64),
                activation=activation, normalize_tx=normalize_tx
            )
            xi = xi_all[i, 0]
            xi = jnp.nan_to_num(xi, nan=0.0, posinf=0.0, neginf=0.0)
            xi = xi_clip * jnp.tanh(xi / xi_clip)
            return xi

        # Derivative helpers (w.r.t. scalar t or x)
        def _tau_t_scalar(params_gen, i, t, x):
            return jax.grad(lambda tt: _tau_i_scalar(params_gen, i, tt, x))(t)

        def _xi_t_scalar(params_gen, i, t, x):
            return jax.grad(lambda tt: _xi_i_scalar(params_gen, i, tt, x))(t)

        def _xi_x_scalar(params_gen, i, t, x):
            return jax.grad(lambda xx: _xi_i_scalar(params_gen, i, t, xx))(x)

        def _xi_xx_scalar(params_gen, i, t, x):
            return jax.grad(jax.grad(lambda xx: _xi_i_scalar(params_gen, i, t, xx)))(x)

        # Vectorize over (i, t, x)
        v_tau   = jax.vmap(lambda ii, tt, xx: _tau_i_scalar(params_gen, ii, tt, xx))
        v_xi    = jax.vmap(lambda ii, tt, xx: _xi_i_scalar(params_gen, ii, tt, xx))
        v_tau_t = jax.vmap(lambda ii, tt, xx: _tau_t_scalar(params_gen, ii, tt, xx))
        v_xi_t  = jax.vmap(lambda ii, tt, xx: _xi_t_scalar(params_gen, ii, tt, xx))
        v_xi_x  = jax.vmap(lambda ii, tt, xx: _xi_x_scalar(params_gen, ii, tt, xx))
        v_xi_xx = jax.vmap(lambda ii, tt, xx: _xi_xx_scalar(params_gen, ii, tt, xx))

        tau0  = v_tau(i_flat, t_flat, x_flat).reshape(m, B)
        xi0   = v_xi(i_flat, t_flat, x_flat).reshape(m, B)
        tau_t = v_tau_t(i_flat, t_flat, x_flat).reshape(m, B)
        xi_t  = v_xi_t(i_flat, t_flat, x_flat).reshape(m, B)
        xi_x  = v_xi_x(i_flat, t_flat, x_flat).reshape(m, B)
        xi_xx = v_xi_xx(i_flat, t_flat, x_flat).reshape(m, B)

        # Clean any non-finites (autodiff can produce NaNs if eval has NaNs)
        tau0  = jnp.nan_to_num(tau0,  nan=0.0, posinf=0.0, neginf=0.0)
        xi0   = jnp.nan_to_num(xi0,   nan=0.0, posinf=0.0, neginf=0.0)
        tau_t = jnp.nan_to_num(tau_t, nan=0.0, posinf=0.0, neginf=0.0)
        xi_t  = jnp.nan_to_num(xi_t,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_x  = jnp.nan_to_num(xi_x,  nan=0.0, posinf=0.0, neginf=0.0)
        xi_xx = jnp.nan_to_num(xi_xx, nan=0.0, posinf=0.0, neginf=0.0)

        return tau0, xi0, tau_t, xi_t, xi_x, xi_xx

    # -------------------- epsilon-flow integrator (unchanged structure) --------------------

    def _flow_heun_allgens(params_gen, t0, x0):
        """
        Push point cloud under each generator, AND integrate prolonged (mu,sig) along epsilon.

        Inputs:
          t0, x0: (B,)
        Returns:
          t_push, x_push, mu_pred, sg_pred: each (m,B)
        """
        tau0, _ = eval_gen_tau_xi(params_gen, t0, x0, activation=activation, normalize_tx=normalize_tx)
        m = tau0.shape[0]
        B = t0.shape[0]

        t_stack = jnp.broadcast_to(t0[None, :], (m, B))
        x_stack = jnp.broadcast_to(x0[None, :], (m, B))

        # initial coefficients on the base cloud
        t0c = jnp.clip(jnp.nan_to_num(t0, nan=0.0, posinf=0.0, neginf=0.0), t_clip_lo, t_clip_hi)
        x0c = jnp.clip(jnp.nan_to_num(x0, nan=0.0, posinf=0.0, neginf=0.0), -x_clip_abs, x_clip_abs)

        mu0 = jnp.nan_to_num(mu_fn(t0c, x0c), nan=0.0, posinf=0.0, neginf=0.0)
        sg0 = jnp.nan_to_num(sig_fn(t0c, x0c), nan=0.0, posinf=0.0, neginf=0.0)
        sg0 = jnp.maximum(jnp.abs(sg0), sigma_floor)

        mu_stack = jnp.broadcast_to(mu0[None, :], (m, B))
        sg_stack = jnp.broadcast_to(sg0[None, :], (m, B))

        def pos_floor(u):
            return sigma_floor + jax.nn.softplus(u - sigma_floor)

        def one_step(_, state):
            tS, xS, muS, sgS = state

            # k1
            tau, xi, tau_t, xi_t, xi_x, xi_xx = _rhs_diag_with_derivs(params_gen, tS, xS)
            k1_t = tau
            k1_x = xi
            k1_mu = xi_t + muS * xi_x + 0.5 * (sgS * sgS) * xi_xx - muS * tau_t
            k1_sg = sgS * xi_x - 0.5 * sgS * tau_t

            t_pred  = tS  + eps * k1_t
            x_pred  = xS  + eps * k1_x
            mu_pred = muS + eps * k1_mu
            sg_pred = pos_floor(sgS + eps * k1_sg)

            # k2
            tau2, xi2, tau_t2, xi_t2, xi_x2, xi_xx2 = _rhs_diag_with_derivs(params_gen, t_pred, x_pred)
            k2_t = tau2
            k2_x = xi2
            k2_mu = xi_t2 + mu_pred * xi_x2 + 0.5 * (sg_pred * sg_pred) * xi_xx2 - mu_pred * tau_t2
            k2_sg = sg_pred * xi_x2 - 0.5 * sg_pred * tau_t2

            t_new  = tS  + 0.5 * eps * (k1_t  + k2_t)
            x_new  = xS  + 0.5 * eps * (k1_x  + k2_x)
            mu_new = muS + 0.5 * eps * (k1_mu + k2_mu)
            sg_new = pos_floor(sgS + 0.5 * eps * (k1_sg + k2_sg))

            return (t_new, x_new, mu_new, sg_new)

        t_stack, x_stack, mu_stack, sg_stack = jax.lax.fori_loop(
            0, num_steps, one_step, (t_stack, x_stack, mu_stack, sg_stack)
        )
        return t_stack, x_stack, mu_stack, sg_stack

    # -------------------- loss (dt-neg penalty removed, stats slot kept) --------------------

    def _loss_impl(params_gen, t_grid, x_paths):
        """
        t_grid:  (N+1,)
        x_paths: (n_traj, N+1)
        """
        t_grid  = jnp.asarray(t_grid,  dtype=jnp.float64)
        x_paths = jnp.asarray(x_paths, dtype=jnp.float64)

        n_traj, Np1 = x_paths.shape

        # ---- Use LEFT endpoints only for coefficient residual (N points per traj) ----
        t_mat  = jnp.broadcast_to(t_grid[None, :], (n_traj, Np1))
        t_left = t_mat[:, :-1].reshape(-1)     # (B_left,)
        x_left = x_paths[:, :-1].reshape(-1)   # (B_left,)
        B_left = t_left.shape[0]

        # Prolonged push from (t_left,x_left,mu(t_left,x_left),sig(t_left,x_left))
        t_push_L, x_push_L, mu_pred, sg_pred = _flow_heun_allgens(params_gen, t_left, x_left)  # each (m,B_left)

        # Evaluate surrogate coefficients at pushed points
        tpc = jnp.clip(jnp.nan_to_num(t_push_L, nan=0.0, posinf=0.0, neginf=0.0), t_clip_lo, t_clip_hi)
        xpc = jnp.clip(jnp.nan_to_num(x_push_L, nan=0.0, posinf=0.0, neginf=0.0), -x_clip_abs, x_clip_abs)

        mu_eval = jnp.nan_to_num(mu_fn(tpc.reshape(-1), xpc.reshape(-1)), nan=0.0, posinf=0.0, neginf=0.0).reshape(-1)
        sg_eval = jnp.nan_to_num(sig_fn(tpc.reshape(-1), xpc.reshape(-1)), nan=0.0, posinf=0.0, neginf=0.0).reshape(-1)

        m = t_push_L.shape[0]
        mu_eval = mu_eval.reshape(m, B_left)
        sg_eval = jnp.maximum(jnp.abs(sg_eval.reshape(m, B_left)), sigma_floor)

        # Per-generator coefficient residuals
        mu_mse = jnp.mean((mu_pred - mu_eval) ** 2, axis=1)  # (m,)
        sg_mse = jnp.mean((sg_pred - sg_eval) ** 2, axis=1)  # (m,)

        # dt-negative penalty intentionally removed; keep slot for compatibility
        dt_neg_mean = jnp.zeros((m,), dtype=jnp.float64)

        per_gen_loss = mu_mse + sg_mse
        loss = jnp.mean(per_gen_loss)

        aux = {
            "per_gen_loss": per_gen_loss,
            # keep same key name and same (m,3) layout: [mu_mse, sg_mse, dt_neg_mean]
            "per_gen_stats": jnp.stack([mu_mse, sg_mse, dt_neg_mean], axis=1),
            "eps": eps,
            "num_steps": jnp.asarray(num_steps, dtype=jnp.int32),
        }
        return loss, aux

    return jax.jit(_loss_impl) if jit else _loss_impl


# Master Loss - SDE Symmetry

In [ ]:
# ============================ Master loss: weighted sum of L1–L7 ============================
# Assumes in scope:
#   - params_sde, in_norm_sde, surrogate_f_sigma
#   - gen_cfg, params_gen
#   - TX_gen  (for sampling tx_batch in the training loop)
#   - make_s1_lie_loss, make_s2_jacobi_loss_nested, make_s3_skewsym_loss,
#     make_s4_bilinearity_loss, make_s5_column_independence_loss,
#     make_s6_commutator_loss_ito, make_s7_pushforward_coeff_loss
#   - t_norm_gen, tx_norm_gen, tau_forward, xi_forward, eval_generators_jit
#
# This cell defines:
#   - LossWeights dataclass (hyperparams)
#   - mu_fn, sig_fn wrapping the neural SDE surrogate
#   - instantiated loss functions s1..s7
#   - master_loss(params_gen, tx_batch, key=None)
#   - master_loss_jit

from dataclasses import dataclass
import jax
import jax.numpy as jnp

# ----------------------- Loss weights / hyperparameters ----------------------

@dataclass
class LossWeights:
    # Algebraic structure terms
    w_s1_closure: float = 1.0   # L1: closure + constancy
    w_s2_jacobi:  float = 0.1   # L2: nested Jacobi identity
    w_s3_skew:    float = 0.1   # L3: skew-symmetry
    w_s4_bilin:   float = 0.1   # L4: bilinearity

    # Column independence
    w_s5_indep:   float = 0.5   # L5: functional independence

    # SDE determining equations
    w_s6_det:     float = 1.0   # L6: Gaeta–Quintero determining equations

    # Finite-ε flow-validity
    w_s7_push:    float = 0.1   # L7: pushforward on (μ, σ)

    # Generator weight decay
    weight_decay: float = 1e-6

    # S5 options
    s5_mode: str = "sigma"      # "sigma" or "corr_l2"
    s5_tau:  float = 0.8

    # S7 options
    s7_eps:   float = 1e-2
    s7_steps: int   = 1

loss_cfg = LossWeights()

# ----------------------- Drift / diffusion wrappers (μ, σ) -------------------

MU_CLIP  = 50.0
SIG_CLIP = 50.0

def mu_fn(t, x):
    f_hat, _ = surrogate_f_sigma(params_sde, in_norm_sde, t, x)
    f_hat = jnp.nan_to_num(f_hat, nan=0.0, posinf=0.0, neginf=0.0)
    f_hat = jnp.clip(f_hat, -MU_CLIP, MU_CLIP)
    return f_hat

def sig_fn(t, x):
    _, sigma_hat = surrogate_f_sigma(params_sde, in_norm_sde, t, x)
    sigma_hat = jnp.nan_to_num(sigma_hat, nan=0.0, posinf=0.0, neginf=0.0)
    sigma_hat = jnp.clip(sigma_hat, -SIG_CLIP, SIG_CLIP)
    return sigma_hat


# ----------------------- Instantiate per-term loss functions -----------------

# L1: Lie bracket closure + constancy
s1_lie_loss = make_s1_lie_loss(n_generators=gen_cfg.n_generators)

# L2: Jacobi identity (nested brackets)
s2_jacobi_loss = make_s2_jacobi_loss_nested(n_generators=gen_cfg.n_generators)

# L3: Skew-symmetry
s3_skew_loss = make_s3_skewsym_loss(n_generators=gen_cfg.n_generators)

# L4: Bilinearity (random coefficient pairs)
s4_bilin_loss = make_s4_bilinearity_loss(
    n_generators=gen_cfg.n_generators,
    num_cc=4,
    cc_list=None,
    normalize=True,
)

# L5: Column independence (functional independence of generators)
s5_indep_loss = make_s5_column_independence_loss(
    n_generators=gen_cfg.n_generators,
    mode=loss_cfg.s5_mode,
    tau=loss_cfg.s5_tau,
    eps=1e-12,
)

# L6: SDE determining equations (Gaeta–Quintero)
s6_det_loss = make_s6_commutator_loss_ito(
    mu_fn=mu_fn,
    sig_fn=sig_fn,
    use_abs=False,   # L2-style penalty
)

# L7: Finite-ε pushforward validity on (μ, σ)
s7_push_loss_raw = make_s7_pushforward_coeff_loss_sde_only(
    mu_fn=mu_fn,
    sig_fn=sig_fn,
    eps=1e-2,
    num_steps=1,
    sigma_floor=1e-6,
    dt_floor=1e-10,
    dt_neg_penalty=10.0,
    activation=gen_cfg.activation,
    normalize_tx=None,   # <-- use normalizer function here
    jit=True,
    tau_clip=5.0,
    xi_clip=5.0,
    x_clip_abs=25.0,
)

# Wrap it so Master Loss can call it as (params_gen, tx_batch)
# ----------------------- S7 data (auto-define once, no extra cell) -----------------------
if ("t_s7" not in globals()) or ("x_paths_s7" not in globals()):
    # pick a deterministic small slice for S7
    n_traj_use = int(min(64, x_surr.shape[0]))
    Np1_use    = int(min(401, x_surr.shape[1]))  # 400 increments

    t_s7 = jnp.asarray(t_surr[:Np1_use], dtype=jnp.float64)                 # (T,)
    x_paths_s7 = jnp.asarray(x_surr[:n_traj_use, :Np1_use], dtype=jnp.float64)  # (n_traj,T)
    globals().update({"t_s7": t_s7, "x_paths_s7": x_paths_s7})

def s7_push_loss(params_gen, tx_batch):
    return s7_push_loss_raw(params_gen, t_s7, x_paths_s7)



# ----------------------- Helper: L2 norm over a pytree -----------------------

def l2_tree(params):
    return sum(jnp.sum(jnp.square(p)) for p in jax.tree_util.tree_leaves(params))

# ----------------------- Master loss ----------------------------------------

def master_loss(params_gen, tx_batch: jnp.ndarray, key=None):
    """
    Compute weighted sum of L1–L7 plus weight decay.

    Args:
      params_gen: generator parameters {"tau": [...], "xi": [...]}
      tx_batch:   (B,2) array of (t,x) points used for all losses.
      key:        optional PRNGKey for L4 (bilinearity) coefficients; if None,
                  L4 uses its own default key each call.

    Returns:
      total_loss, aux where aux is a dict with per-term components.
    """
    total = 0.0
    aux = {}

    # L1: closure + constancy
    loss_s1, aux_s1 = s1_lie_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s1_closure * loss_s1
    aux["L1"] = {"loss": loss_s1, **aux_s1}

    # L2: Jacobi
    loss_s2, aux_s2 = s2_jacobi_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s2_jacobi * loss_s2
    aux["L2"] = {"loss": loss_s2, **aux_s2}

    # L3: skew-symmetry
    loss_s3, aux_s3 = s3_skew_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s3_skew * loss_s3
    aux["L3"] = {"loss": loss_s3, **aux_s3}

    # L4: bilinearity (needs key)
    loss_s4, aux_s4 = s4_bilin_loss(params_gen, tx_batch, key=key)
    total = total + loss_cfg.w_s4_bilin * loss_s4
    aux["L4"] = {"loss": loss_s4, **aux_s4}

    # L5: column independence
    loss_s5, aux_s5 = s5_indep_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s5_indep * loss_s5
    aux["L5"] = {"loss": loss_s5, **aux_s5}

    # L6: SDE determining equations
    loss_s6, aux_s6 = s6_det_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s6_det * loss_s6
    aux["L6"] = {"loss": loss_s6, **aux_s6}

    # L7: pushforward (flow-validity)
    loss_s7, aux_s7 = s7_push_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s7_push * loss_s7
    aux["L7"] = {"loss": loss_s7, **aux_s7}

    # Weight decay on generator parameters
    wd = loss_cfg.weight_decay * l2_tree(params_gen)
    total = total + wd
    aux["weight_decay"] = wd

    aux["total"] = total
    return total, aux

master_loss_jit = jax.jit(master_loss)

# Export to globals so the training cell can just call master_loss_jit
globals().update({
    "loss_cfg": loss_cfg,
    "master_loss": master_loss,
    "master_loss_jit": master_loss_jit,
})


# Training

In [ ]:
# ============================ Generator training loop (Stage 2) ============================
from dataclasses import dataclass
import numpy as np
import jax
import jax.numpy as jnp

# ---------- training config ----------
@dataclass
class GenTrainConfig:
    steps: int = 3000
    batch_size: int = 2048
    lr: float = 1e-3
    print_every: int = 100

gen_train_cfg = GenTrainConfig()

# ---------- re-init params_gen if any int32 leaves leaked in (grad can't handle ints) ----------
def _reinit_params_gen(key, gen_cfg):
    assert "init_mlp_params" in globals(), "Missing init_mlp_params."
    m = int(getattr(gen_cfg, "n_generators"))
    hidden_tau  = int(getattr(gen_cfg, "hidden_tau", 32))
    hidden_xi   = int(getattr(gen_cfg, "hidden_xi", 64))

    keys = jax.random.split(key, 2 * m)
    params_tau_list, params_xi_list = [], []

    for i in range(m):
        k_tau  = keys[2 * i]
        k_xi   = keys[2 * i + 1]

        sizes_tau = [1] + [gen_cfg.hidden_tau] * gen_cfg.depth_tau + [1]
        sizes_xi  = [2] + [gen_cfg.hidden_xi]  * gen_cfg.depth_xi  + [1]

        params_tau = init_mlp_params(k_tau, sizes=sizes_tau)
        params_xi  = init_mlp_params(k_xi,  sizes=sizes_xi)

        params_tau_list.append(params_tau)
        params_xi_list.append(params_xi)

    return {"tau": params_tau_list, "xi": params_xi_list}

_leaves = jax.tree_util.tree_leaves(params_gen)
_has_int = any(isinstance(a, jnp.ndarray) and jnp.issubdtype(a.dtype, jnp.integer) for a in _leaves)
if _has_int:
    key_main, key_gen = jax.random.split(key_main)
    params_gen = _reinit_params_gen(key_gen, gen_cfg)

# ---------- optimizer ----------
assert "optax" in globals(), "optax must be imported."
gen_train_cfg.lr = 1e-4  # <-- LR

optimizer_gen = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(gen_train_cfg.lr),
)

opt_state_gen = optimizer_gen.init(params_gen)

# ---------- minibatch sampler ----------
TX_gen_np = np.asarray(TX_gen, dtype=np.float64)
mask = np.isfinite(TX_gen_np).all(axis=1)
TX_gen_np = TX_gen_np[mask]
N_tx = TX_gen_np.shape[0]
assert N_tx > 0, "All TX_gen points are non-finite after filtering."

rng_np_gen = np.random.default_rng(42)

def sample_tx_batch(batch_size: int):
    if batch_size >= N_tx:
        idx = np.arange(N_tx)
    else:
        idx = rng_np_gen.choice(N_tx, size=batch_size, replace=False)
    return jnp.asarray(TX_gen_np[idx], dtype=jnp.float64)

# ---------- one JIT-ed training step ----------
@jax.jit
def train_step(params_gen, opt_state_gen, tx_batch, key):
    def loss_for_grad(p):
        loss_val, aux = master_loss(p, tx_batch, key)
        return loss_val, aux

    (loss_val, aux), grads = jax.value_and_grad(loss_for_grad, has_aux=True)(params_gen)
    updates, opt_state_new = optimizer_gen.update(grads, opt_state_gen, params_gen)
    params_new = optax.apply_updates(params_gen, updates)
    return params_new, opt_state_new, loss_val, aux

# ---------- helpers for logging ----------
LOSS_KEYS = ["L1", "L2", "L3", "L4", "L5", "L6", "L7"]

def _safe_float(x):
    # works for scalars and 0-d arrays
    return float(jnp.asarray(x))

def format_log(step, total_loss, aux):
    # Pull per-term losses if present
    parts = [f"step {step:5d}/{gen_train_cfg.steps}", f"total={_safe_float(total_loss):.6e}"]
    for k in LOSS_KEYS:
        if k in aux and "loss" in aux[k]:
            parts.append(f"{k}={_safe_float(aux[k]['loss']):.3e}")
    if "weight_decay" in aux:
        parts.append(f"wd={_safe_float(aux['weight_decay']):.3e}")
    return " | ".join(parts)

# ---------- training loop ----------
loss_history_gen = []
loss_hist = {k: [] for k in LOSS_KEYS}
wd_history = []

key_main, key_train = jax.random.split(key_main, 2)

print(f"Starting generator training for {gen_train_cfg.steps} steps "
      f"with batch_size={gen_train_cfg.batch_size}, lr={gen_train_cfg.lr}")

for step in range(1, gen_train_cfg.steps + 1):
    tx_batch = sample_tx_batch(gen_train_cfg.batch_size)
    key_train, key_step = jax.random.split(key_train)

    params_gen, opt_state_gen, loss_val, aux = train_step(params_gen, opt_state_gen, tx_batch, key_step)

    loss_history_gen.append(_safe_float(loss_val))

    for k in LOSS_KEYS:
        if k in aux and "loss" in aux[k]:
            loss_hist[k].append(_safe_float(aux[k]["loss"]))
        else:
            loss_hist[k].append(np.nan)

    wd_history.append(_safe_float(aux.get("weight_decay", 0.0)))

    if (step % gen_train_cfg.print_every == 0) or (step == 1) or (step == gen_train_cfg.steps):
        print(format_log(step, loss_val, aux))

print("\nGenerator training finished.")

globals().update({
    "params_gen": params_gen,
    "opt_state_gen": opt_state_gen,
    "loss_history_gen": loss_history_gen,
    "loss_hist": loss_hist,          # dict: keys L1..L7 -> list
    "wd_history": wd_history,
    "gen_train_cfg": gen_train_cfg,
})


#SDE-sym. Evaluations

## Heat maps

In [ ]:
# === Evaluation 1: visualize learned generators (τ_i(t) and ξ_i(t,x)) ===
# Compatible with eval_generators_jit returning (tau, xi, beta) or (tau, xi, beta, phi) etc.

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

m = gen_cfg.n_generators

# -------- user-adjustable grid settings --------------------------------------
t_min = float(t_surr.min())
t_max = float(t_surr.max())
x_min = float(x_surr.min())
x_max = float(x_surr.max())

Nt_line = 200   # for τ(t) 1D plots
Nt = 50         # for ξ(t,x) heatmaps (time resolution)
Nx = 50         # for ξ(t,x) heatmaps (space resolution)

print(f"Using t in [{t_min:.3f}, {t_max:.3f}], x in [{x_min:.3f}, {x_max:.3f}]")
print(f"Heatmap grid: Nt={Nt}, Nx={Nx}")

# -------- helper: robustly extract (tau, xi) from eval_generators_jit ----------
def _eval_tau_xi(params_gen, t, x):
    out = eval_generators_jit(params_gen, t, x)
    # out could be (tau, xi) or (tau, xi, beta) or (tau, xi, beta, phi)
    if isinstance(out, (tuple, list)):
        if len(out) < 2:
            raise ValueError(f"eval_generators_jit returned tuple/list of len {len(out)}; expected >=2")
        tau, xi = out[0], out[1]
    else:
        raise ValueError("eval_generators_jit returned non-tuple; expected (tau, xi, ...)")
    return tau, xi

# -------- 1D τ_i(t) curves ---------------------------------------------------
t_line = jnp.linspace(t_min, t_max, Nt_line)
x_zero = jnp.zeros_like(t_line)  # τ_i(t) independent of x by construction

tau_vals_line, _ = _eval_tau_xi(params_gen, t_line, x_zero)  # (m, Nt_line)

plt.figure(figsize=(7, 2.5 * m))
for i in range(m):
    plt.subplot(m, 1, i + 1)
    plt.plot(np.asarray(t_line), np.asarray(tau_vals_line[i]), lw=2)
    plt.xlabel("t")
    plt.ylabel(rf"$\tau_{i+1}(t)$")
    plt.title(rf"Generator {i+1}: $\tau_{i+1}(t)$")
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# -------- 2D ξ_i(t,x) heatmaps ----------------------------------------------
t_eval = jnp.linspace(t_min, t_max, Nt)
x_eval = jnp.linspace(x_min, x_max, Nx)
TT, XX = jnp.meshgrid(t_eval, x_eval, indexing="ij")  # (Nt,Nx)

t_flat = TT.reshape(-1)
x_flat = XX.reshape(-1)

_, xi_vals_flat = _eval_tau_xi(params_gen, t_flat, x_flat)  # (m, Nt*Nx)
xi_grids = xi_vals_flat.reshape(m, Nt, Nx)

fig, axes = plt.subplots(m, 1, figsize=(8, 3 * m), constrained_layout=True)
if m == 1:
    axes = [axes]

for i in range(m):
    ax = axes[i]
    im = ax.imshow(
        np.asarray(xi_grids[i]),
        origin="lower",
        extent=(float(x_eval.min()), float(x_eval.max()),
                float(t_eval.min()), float(t_eval.max())),
        aspect="auto"
    )
    ax.set_title(rf"Generator {i+1}: $\xi_{i+1}(t,x)$")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    fig.colorbar(im, ax=ax, shrink=0.9)

plt.show()


## Span Check

In [ ]:
# === Evaluation 2: principal angles (variable m) against ground truth basis ===
# Uses the *same (t,x) point cloud that Stage-2 generator training samples from* (TX_gen),
# instead of a separate (t,x) meshgrid.
#
# Ground truth — SDE symmetry generators:
#   v1 = ∂_t                                   -> (τ=1,      ξ=0)
#   v3 = e^t ∂_x                               -> (τ=0,      ξ=e^t)
#   v5 = e^{2t}[ ∂_t + x∂_x - u∂_u ]            -> (τ=e^{2t}, ξ=e^{2t} x)
#
# NOTE: span-check uses only (τ, ξ), so the u∂_u part is dropped in v5.

import itertools
import jax
import jax.numpy as jnp
import numpy as np

# -------- helper: robustly extract (tau, xi) from eval_generators_jit ----------
def _eval_tau_xi(params_gen, t, x):
    out = eval_generators_jit(params_gen, t, x)
    if isinstance(out, (tuple, list)):
        if len(out) < 2:
            raise ValueError(f"eval_generators_jit returned tuple/list of len {len(out)}; expected >=2")
        tau, xi = out[0], out[1]
    else:
        raise ValueError("eval_generators_jit returned non-tuple; expected (tau, xi, ...)")
    return tau, xi

# -----------------------------------------------------------------------------
# IMPORTANT CHANGE: evaluation points = training point cloud (TX_gen)
# -----------------------------------------------------------------------------
if "TX_gen" not in globals():
    raise NameError("TX_gen not found. Run the Stage-2 data generation cell first.")

TX_eval_np = np.asarray(TX_gen, dtype=np.float64)
mask = np.isfinite(TX_eval_np).all(axis=1)
TX_eval_np = TX_eval_np[mask]

B_all = TX_eval_np.shape[0]
if B_all == 0:
    raise ValueError("No finite TX_gen points available for evaluation.")

# Optional safety cap (keeps evaluation from getting too heavy); still samples ONLY from training points.
MAX_B_EVAL = int(25000)  # set to None to force using all points
if (MAX_B_EVAL is not None) and (B_all > MAX_B_EVAL):
    rng_eval = np.random.default_rng(0)
    idx = rng_eval.choice(B_all, size=MAX_B_EVAL, replace=False)
    TX_eval_np = TX_eval_np[idx]
    print(f"Principal-angle eval: subsampled {MAX_B_EVAL} points from training cloud (original B={B_all}).")
else:
    print(f"Principal-angle eval: using ALL finite training points B={B_all}.")

t_flat = jnp.asarray(TX_eval_np[:, 0], dtype=jnp.float64)
x_flat = jnp.asarray(TX_eval_np[:, 1], dtype=jnp.float64)
B = t_flat.shape[0]
print("Total evaluation points B =", int(B))

# -------- learned generators (n = gen_cfg.n_generators) -----------------------
n = int(gen_cfg.n_generators)
if n <= 0:
    raise ValueError(f"Expected gen_cfg.n_generators >= 1, got {n}")

tau_learn, xi_learn = _eval_tau_xi(params_gen, t_flat, x_flat)  # (n, B)

v_cols_all = []
for i in range(n):
    vec_i = jnp.concatenate([tau_learn[i], xi_learn[i]], axis=0)  # (2B,)
    v_cols_all.append(vec_i)
V_all = jnp.stack(v_cols_all, axis=1)   # (2B, n)

# -------- ground-truth generators (3) ----------------------------------------
# v1: τ=1, ξ=0
w1_tau = jnp.ones_like(t_flat)
w1_xi  = jnp.zeros_like(x_flat)
w1_vec = jnp.concatenate([w1_tau, w1_xi], axis=0)

# v3: τ=0, ξ=e^t
w2_tau = jnp.zeros_like(t_flat)
w2_xi  = jnp.exp(t_flat)
w2_vec = jnp.concatenate([w2_tau, w2_xi], axis=0)

# v5: τ=e^{2t}, ξ=e^{2t} x
w3_tau = jnp.exp(2.0 * t_flat)
w3_xi  = jnp.exp(2.0 * t_flat) * x_flat
w3_vec = jnp.concatenate([w3_tau, w3_xi], axis=0)

W_all = jnp.stack([w1_vec, w2_vec, w3_vec], axis=1)    # (2B, 3)
gt_labels = ["v1(∂t)", "v3(e^t∂x)", "v5(e^{2t}(∂t+x∂x- u∂u))"]

# -------- principal angles ----------------------------------------------------
def principal_angles(V, W):
    Q1, _ = jnp.linalg.qr(V, mode="reduced")
    Q2, _ = jnp.linalg.qr(W, mode="reduced")
    s = jnp.linalg.svd(Q1.T @ Q2, compute_uv=False)
    s = jnp.clip(s, -1.0, 1.0)
    return jnp.sort(jnp.arccos(s))

def cos_sim(a, b):
    return float((a @ b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b) + 1e-12))

# -------- comparison logic per n vs 3 ----------------------------------------
if n == 3:
    V = V_all
    W = W_all

    angles_rad = principal_angles(V, W)
    angles_deg = angles_rad * (180.0 / jnp.pi)

    print("\nPrincipal angles between learned span{X_i} and ground-truth span{v1, v3, v5}:")
    for k, (ang_r, ang_d) in enumerate(zip(angles_rad, angles_deg), start=1):
        print(f"  angle {k}: {float(ang_r):.6f} rad  = {float(ang_d):.4f} degrees")

    print("\nPairwise cosine similarities (learned i vs ground-truth j):")
    for i in range(3):
        for j in range(3):
            cij = cos_sim(V[:, i], W[:, j])
            print(f"  <X_{i+1}, v_{j+1}> = {cij:.4f}")

elif n > 3:
    combos = list(itertools.combinations(range(n), 3))
    print(f"\nLearned generators n={n} > 3: evaluating (n choose 3) = {len(combos)} learned 3-subsets vs full ground-truth 3D basis.")

    for c_idx, c in enumerate(combos, start=1):
        cols = list(c)
        V = V_all[:, cols]   # (2B, 3)
        W = W_all            # (2B, 3)

        angles_rad = principal_angles(V, W)
        angles_deg = angles_rad * (180.0 / jnp.pi)

        pretty_subset = ", ".join([f"X_{i+1}" for i in cols])
        print(f"\n[{c_idx}/{len(combos)}] subset {{{pretty_subset}}} vs ground truth {{{', '.join(gt_labels)}}}:")
        for k, (ang_r, ang_d) in enumerate(zip(angles_rad, angles_deg), start=1):
            print(f"  angle {k}: {float(ang_r):.6f} rad  = {float(ang_d):.4f} degrees")

        print("  Pairwise cosine similarities (subset learned a vs ground-truth j):")
        for a in range(3):
            for j in range(3):
                cij = cos_sim(V[:, a], W[:, j])
                print(f"    <{pretty_subset.split(', ')[a]}, {gt_labels[j]}> = {cij:.4f}")

else:  # n < 3
    combos = list(itertools.combinations(range(3), n))
    print(f"\nLearned generators n={n} < 3: evaluating (3 choose n) = {len(combos)} ground-truth n-subsets vs learned nD span.")

    V = V_all  # (2B, n)
    for c_idx, c in enumerate(combos, start=1):
        gt_cols = list(c)
        W = W_all[:, gt_cols]  # (2B, n)

        angles_rad = principal_angles(V, W)
        angles_deg = angles_rad * (180.0 / jnp.pi)

        pretty_gt = ", ".join([gt_labels[i] for i in gt_cols])
        print(f"\n[{c_idx}/{len(combos)}] learned {{{', '.join([f'X_{i+1}' for i in range(n)])}}} vs ground truth subset {{{pretty_gt}}}:")
        for k, (ang_r, ang_d) in enumerate(zip(angles_rad, angles_deg), start=1):
            print(f"  angle {k}: {float(ang_r):.6f} rad  = {float(ang_d):.4f} degrees")

        print("  Pairwise cosine similarities (learned i vs selected ground-truth j):")
        for i in range(n):
            for jj, j in enumerate(gt_cols):
                cij = cos_sim(V[:, i], W_all[:, j])
                print(f"    <X_{i+1}, {gt_labels[j]}> = {cij:.4f}")


In [ ]:
# === Evaluation 2: principal angles (variable m) against ground truth basis ===
# NOW: samples evaluation points uniformly from a user-controlled (t,x) rectangle,
# instead of using TX_gen.
#
# Ground truth — SDE symmetry generators:
#   v1 = ∂_t                                   -> (τ=1,      ξ=0)
#   v3 = e^t ∂_x                               -> (τ=0,      ξ=e^t)
#   v5 = e^{2t}[ ∂_t + x∂_x - u∂_u ]            -> (τ=e^{2t}, ξ=e^{2t} x)
#
# NOTE: span-check uses only (τ, ξ), so the u∂_u part is dropped in v5.

import itertools
import jax
import jax.numpy as jnp
import numpy as np

# -------- helper: robustly extract (tau, xi) from eval_generators_jit ----------
def _eval_tau_xi(params_gen, t, x):
    out = eval_generators_jit(params_gen, t, x)
    if isinstance(out, (tuple, list)):
        if len(out) < 2:
            raise ValueError(f"eval_generators_jit returned tuple/list of len {len(out)}; expected >=2")
        tau, xi = out[0], out[1]
    else:
        raise ValueError("eval_generators_jit returned non-tuple; expected (tau, xi, ...)")
    return tau, xi

# -----------------------------------------------------------------------------
# NEW: evaluation points sampled uniformly from a user-controlled rectangle
# -----------------------------------------------------------------------------
# ---- user knobs ----
T_EVAL_LO = 0.0
T_EVAL_HI = 0.5
X_EVAL_LO = -1.0
X_EVAL_HI =  2.0

B_EVAL = int(25000)   # number of eval points
EVAL_SEED = 0         # reproducible sampling seed
# --------------------

if B_EVAL <= 0:
    raise ValueError(f"B_EVAL must be positive, got {B_EVAL}")
if not (T_EVAL_HI > T_EVAL_LO):
    raise ValueError("Need T_EVAL_HI > T_EVAL_LO")
if not (X_EVAL_HI > X_EVAL_LO):
    raise ValueError("Need X_EVAL_HI > X_EVAL_LO")

rng_eval = np.random.default_rng(int(EVAL_SEED))
t_np = rng_eval.uniform(float(T_EVAL_LO), float(T_EVAL_HI), size=B_EVAL).astype(np.float64)
x_np = rng_eval.uniform(float(X_EVAL_LO), float(X_EVAL_HI), size=B_EVAL).astype(np.float64)

mask = np.isfinite(t_np) & np.isfinite(x_np)
t_np = t_np[mask]
x_np = x_np[mask]
B = int(t_np.shape[0])
if B == 0:
    raise ValueError("No finite (t,x) evaluation points after filtering.")

t_flat = jnp.asarray(t_np, dtype=jnp.float64)
x_flat = jnp.asarray(x_np, dtype=jnp.float64)
print(f"Principal-angle eval: uniform sample B={B} from t∈[{T_EVAL_LO},{T_EVAL_HI}] x∈[{X_EVAL_LO},{X_EVAL_HI}]")

# -------- learned generators (n = gen_cfg.n_generators) -----------------------
n = int(gen_cfg.n_generators)
if n <= 0:
    raise ValueError(f"Expected gen_cfg.n_generators >= 1, got {n}")

tau_learn, xi_learn = _eval_tau_xi(params_gen, t_flat, x_flat)  # (n, B)

v_cols_all = []
for i in range(n):
    vec_i = jnp.concatenate([tau_learn[i], xi_learn[i]], axis=0)  # (2B,)
    v_cols_all.append(vec_i)
V_all = jnp.stack(v_cols_all, axis=1)   # (2B, n)

# -------- ground-truth generators (3) ----------------------------------------
# v1: τ=1, ξ=0
w1_tau = jnp.ones_like(t_flat)
w1_xi  = jnp.zeros_like(x_flat)
w1_vec = jnp.concatenate([w1_tau, w1_xi], axis=0)

# v3: τ=0, ξ=e^t
w2_tau = jnp.zeros_like(t_flat)
w2_xi  = jnp.exp(t_flat)
w2_vec = jnp.concatenate([w2_tau, w2_xi], axis=0)

# v5: τ=e^{2t}, ξ=e^{2t} x
w3_tau = jnp.exp(2.0 * t_flat)
w3_xi  = jnp.exp(2.0 * t_flat) * x_flat
w3_vec = jnp.concatenate([w3_tau, w3_xi], axis=0)

W_all = jnp.stack([w1_vec, w2_vec, w3_vec], axis=1)    # (2B, 3)
gt_labels = ["v1(∂t)", "v3(e^t∂x)", "v5(e^{2t}(∂t+x∂x- u∂u))"]

# -------- principal angles ----------------------------------------------------
def principal_angles(V, W):
    Q1, _ = jnp.linalg.qr(V, mode="reduced")
    Q2, _ = jnp.linalg.qr(W, mode="reduced")
    s = jnp.linalg.svd(Q1.T @ Q2, compute_uv=False)
    s = jnp.clip(s, -1.0, 1.0)
    return jnp.sort(jnp.arccos(s))

def cos_sim(a, b):
    return float((a @ b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b) + 1e-12))

# -------- comparison logic per n vs 3 ----------------------------------------
if n == 3:
    V = V_all
    W = W_all

    angles_rad = principal_angles(V, W)
    angles_deg = angles_rad * (180.0 / jnp.pi)

    print("\nPrincipal angles between learned span{X_i} and ground-truth span{v1, v3, v5}:")
    for k, (ang_r, ang_d) in enumerate(zip(angles_rad, angles_deg), start=1):
        print(f"  angle {k}: {float(ang_r):.6f} rad  = {float(ang_d):.4f} degrees")

    print("\nPairwise cosine similarities (learned i vs ground-truth j):")
    for i in range(3):
        for j in range(3):
            cij = cos_sim(V[:, i], W[:, j])
            print(f"  <X_{i+1}, v_{j+1}> = {cij:.4f}")

elif n > 3:
    combos = list(itertools.combinations(range(n), 3))
    print(f"\nLearned generators n={n} > 3: evaluating (n choose 3) = {len(combos)} learned 3-subsets vs full ground-truth 3D basis.")

    for c_idx, c in enumerate(combos, start=1):
        cols = list(c)
        V = V_all[:, cols]   # (2B, 3)
        W = W_all            # (2B, 3)

        angles_rad = principal_angles(V, W)
        angles_deg = angles_rad * (180.0 / jnp.pi)

        pretty_subset = ", ".join([f"X_{i+1}" for i in cols])
        print(f"\n[{c_idx}/{len(combos)}] subset {{{pretty_subset}}} vs ground truth {{{', '.join(gt_labels)}}}:")
        for k, (ang_r, ang_d) in enumerate(zip(angles_rad, angles_deg), start=1):
            print(f"  angle {k}: {float(ang_r):.6f} rad  = {float(ang_d):.4f} degrees")

        print("  Pairwise cosine similarities (subset learned a vs ground-truth j):")
        for a in range(3):
            for j in range(3):
                cij = cos_sim(V[:, a], W[:, j])
                print(f"    <{pretty_subset.split(', ')[a]}, {gt_labels[j]}> = {cij:.4f}")

else:  # n < 3
    combos = list(itertools.combinations(range(3), n))
    print(f"\nLearned generators n={n} < 3: evaluating (3 choose n) = {len(combos)} ground-truth n-subsets vs learned nD span.")

    V = V_all  # (2B, n)
    for c_idx, c in enumerate(combos, start=1):
        gt_cols = list(c)
        W = W_all[:, gt_cols]  # (2B, n)

        angles_rad = principal_angles(V, W)
        angles_deg = angles_rad * (180.0 / jnp.pi)

        pretty_gt = ", ".join([gt_labels[i] for i in gt_cols])
        print(f"\n[{c_idx}/{len(combos)}] learned {{{', '.join([f'X_{i+1}' for i in range(n)])}}} vs ground truth subset {{{pretty_gt}}}:")
        for k, (ang_r, ang_d) in enumerate(zip(angles_rad, angles_deg), start=1):
            print(f"  angle {k}: {float(ang_r):.6f} rad  = {float(ang_d):.4f} degrees")

        print("  Pairwise cosine similarities (learned i vs selected ground-truth j):")
        for i in range(n):
            for jj, j in enumerate(gt_cols):
                cij = cos_sim(V[:, i], W_all[:, j])
                print(f"    <X_{i+1}, {gt_labels[j]}> = {cij:.4f}")


In [ ]:
# === Evaluation 2b: stronger span check (block-balanced + best-mixing residual) ===
# Compatible with upstream: uses eval_generators_jit(params_gen, t, x) and ignores extra returns.

import jax
import jax.numpy as jnp
import numpy as np

# -------- helper: robustly extract (tau, xi) from eval_generators_jit ----------
def _eval_tau_xi(params_gen, t, x):
    out = eval_generators_jit(params_gen, t, x)
    if not isinstance(out, (tuple, list)) or len(out) < 2:
        raise ValueError("eval_generators_jit must return (tau, xi, ...) with at least 2 outputs.")
    return out[0], out[1]

# -------- helper: per-generator block balancing (tau/xi comparable energy) ----
def _stack_balanced(tau, xi, eps=1e-12):
    """
    tau, xi: (m, B)
    Returns V: (2B, m) where each column i has tau and xi separately standardized to unit RMS.
    """
    # RMS per generator per block
    tau_rms = jnp.sqrt(jnp.mean(tau**2, axis=1, keepdims=True) + eps)
    xi_rms  = jnp.sqrt(jnp.mean(xi**2,  axis=1, keepdims=True) + eps)
    tau_s = tau / tau_rms
    xi_s  = xi  / xi_rms
    V = jnp.concatenate([tau_s, xi_s], axis=1).T  # (2B, m)
    return V

# -------- helper: principal angles between column spans -----------------------
def principal_angles(V, W):
    # V,W: (D,k)
    Q1, _ = jnp.linalg.qr(V, mode="reduced")
    Q2, _ = jnp.linalg.qr(W, mode="reduced")
    s = jnp.linalg.svd(Q1.T @ Q2, compute_uv=False)
    s = jnp.clip(s, -1.0, 1.0)
    return jnp.sort(jnp.arccos(s))

# -------- helper: best-mixing residual ---------------------------------------
def best_mixing_residual(V, W, ridge=1e-10):
    """
    Solve A* = argmin_A ||V - W A||_F (with tiny ridge), report relative residual.
    V,W: (D,k), k is small (e.g. 3)
    """
    # A = (W^T W + ridge I)^{-1} W^T V
    k = W.shape[1]
    WT_W = W.T @ W
    A = jnp.linalg.solve(WT_W + ridge * jnp.eye(k, dtype=W.dtype), W.T @ V)
    Vhat = W @ A
    rel = jnp.linalg.norm(V - Vhat) / (jnp.linalg.norm(V) + 1e-12)
    return rel, A

# -------- choose evaluation points: prefer empirical TX_gen if available -------
USE_TX_GEN = True
B_max = 5000  # cap for speed

if USE_TX_GEN and ("TX_gen" in globals()):
    TX = jnp.asarray(TX_gen, dtype=jnp.float64)
    TX = TX[jnp.isfinite(TX).all(axis=1)]
    if TX.shape[0] == 0:
        raise ValueError("TX_gen is empty after filtering non-finites.")
    B = int(min(B_max, TX.shape[0]))
    t_flat = TX[:B, 0]
    x_flat = TX[:B, 1]
    print(f"[span-check] Using empirical TX_gen points: B={B}")
    print("t range:", float(t_flat.min()), float(t_flat.max()))
    print("x range:", float(x_flat.min()), float(x_flat.max()))

else:
    # fallback to a uniform grid using t_surr/x_surr if present
    t_min_eval = float(t_surr.min()) if "t_surr" in globals() else 0.0
    t_max_eval = float(t_surr.max()) if "t_surr" in globals() else 5.0
    x_min_eval = float(x_surr.min()) if "x_surr" in globals() else -4.0
    x_max_eval = float(x_surr.max()) if "x_surr" in globals() else 6.0
    Nt_eval, Nx_eval = 40, 40
    t_eval = jnp.linspace(t_min_eval, t_max_eval, Nt_eval)
    x_eval = jnp.linspace(x_min_eval, x_max_eval, Nx_eval)
    TT, XX = jnp.meshgrid(t_eval, x_eval, indexing="ij")
    t_flat = TT.reshape(-1)
    x_flat = XX.reshape(-1)
    print("t range:", float(t_flat.min()), float(t_flat.max()))
    print("x range:", float(x_flat.min()), float(x_flat.max()))

    B = int(t_flat.shape[0])
    print(f"[span-check] Using uniform grid: B={B} on t∈[{t_min_eval},{t_max_eval}], x∈[{x_min_eval},{x_max_eval}]")

# -------- learned generators --------------------------------------------------
m = int(gen_cfg.n_generators)
tau_learn, xi_learn = _eval_tau_xi(params_gen, t_flat, x_flat)  # (m,B)

# -------- ground-truth subspace for m=3 check --------------------------------
# Change ONLY the ground-truth basis to current {v1, v3, v5}:
#   v1 = ∂t                 -> (τ=1,        ξ=0)
#   v3 = e^t ∂x             -> (τ=0,        ξ=e^t)
#   v5 = e^{2t}(∂t + x∂x)   -> (τ=e^{2t},   ξ=e^{2t} x)
w1_tau = jnp.ones_like(t_flat)
w1_xi  = jnp.zeros_like(x_flat)

w2_tau = jnp.zeros_like(t_flat)
w2_xi  = jnp.exp(t_flat)

w3_tau = jnp.exp(2.0 * t_flat)
w3_xi  = jnp.exp(2.0 * t_flat) * x_flat

W_tau = jnp.stack([w1_tau, w2_tau, w3_tau], axis=0)  # (3,B)
W_xi  = jnp.stack([w1_xi,  w2_xi,  w3_xi ], axis=0)  # (3,B)

# -------- build balanced stacked matrices V and W -----------------------------
# For learned: take first k=3 generators if m>3 (consistent with m=3 span check)
k = 3
if m < k:
    raise ValueError(f"Need at least {k} learned generators for this check; got m={m}.")

V_bal = _stack_balanced(tau_learn[:k], xi_learn[:k])   # (2B, k)
W_bal = _stack_balanced(W_tau, W_xi)                   # (2B, k)

# Also compute the "raw" (unbalanced) version for comparison
V_raw = jnp.concatenate([tau_learn[:k], xi_learn[:k]], axis=1).T  # (2B,k)
W_raw = jnp.concatenate([W_tau, W_xi], axis=1).T                  # (2B,k)

# -------- principal angles (raw vs balanced) ---------------------------------
angles_raw = principal_angles(V_raw, W_raw)
angles_bal = principal_angles(V_bal, W_bal)

def _deg(x): return x * (180.0 / jnp.pi)

print("\nPrincipal angles (RAW stacking):")
for i, a in enumerate(angles_raw, 1):
    print(f"  angle {i}: {float(a):.6f} rad = {float(_deg(a)):.4f} deg")

print("\nPrincipal angles (BALANCED tau/xi per generator):")
for i, a in enumerate(angles_bal, 1):
    print(f"  angle {i}: {float(a):.6f} rad = {float(_deg(a)):.4f} deg")

# -------- best-mixing residual (raw vs balanced) ------------------------------
rel_raw, A_raw = best_mixing_residual(V_raw, W_raw)
rel_bal, A_bal = best_mixing_residual(V_bal, W_bal)

print("\nBest-mixing relative residuals (lower is better):")
print(f"  RAW:      ||V - W A*|| / ||V|| = {float(rel_raw):.6e}")
print(f"  BALANCED: ||V - W A*|| / ||V|| = {float(rel_bal):.6e}")

print("\nBest-mixing matrix A* (BALANCED), columns correspond to learned generators in V:")
print(np.asarray(A_bal))

# -------- optional: pairwise cosines after balancing --------------------------
def cos_sim(a, b):
    return float((a @ b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b) + 1e-12))

print("\nPairwise cosine similarities using BALANCED stacked columns:")
for i in range(k):
    for j in range(k):
        cij = cos_sim(V_bal[:, i], W_bal[:, j])
        print(f"  <X_{i+1}, v_{j+1}> = {cij:.4f}")
